# SIR Graph EMM — Final Reproducible Experiment Notebook

This  notebook contains the final CBS-calibrated household model, layered contact network, transmission-probability calibration, cross-network Model A/Model B validation, and mechanism stress tests.

**Reproducibility check:** restart the kernel and run all cells from top to bottom. 


## 1. Imports


In [48]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from IPython.display import display

from sir_graph.dynamics import simulate
from sir_graph.graph import Graph
from sir_graph.emm import (
    search_subgroups,
    search_model_b_subgroups,
    score_model_b_subgroup,
    search_model_a_subgroups,
    score_model_a_subgroup,
)
from sir_graph.parameters import NodeParams
from sir_graph.analysis import (
    model_b_observations,
    model_a_observations,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Population and household model


In [3]:
# ============================================================
# Population and household-generation constants
# ============================================================

N_NODES = 10_000
DATA_SEED = 22
HOUSEHOLD_SEED = 20261020


# ============================================================
# Initial household-size assignment
# ============================================================

def assign_households(
    people,
    *,
    seed,
):
    """
    Generate household IDs and household sizes.

    Household sizes are sampled from the global
    HOUSEHOLD_SIZES / HOUSEHOLD_SIZE_PROBS distribution.

    The age-aware reassignment performed later preserves
    these generated household sizes exactly.
    """

    rng = np.random.default_rng(
        seed
    )

    result = people.copy()

    n_people = len(
        result
    )

    household_id = np.full(
        n_people,
        -1,
        dtype=np.int64,
    )

    household_size = np.zeros(
        n_people,
        dtype=np.int64,
    )

    # Random order in which people are assigned
    # to generated households.
    person_order = rng.permutation(
        n_people
    )

    cursor = 0
    current_household = 0

    while cursor < n_people:

        remaining = (
            n_people
            - cursor
        )

        size = int(
            rng.choice(
                HOUSEHOLD_SIZES,
                p=HOUSEHOLD_SIZE_PROBS,
            )
        )

        # Final household must not exceed
        # the remaining population.
        size = min(
            size,
            remaining,
        )

        members = person_order[
            cursor:
            cursor + size
        ]

        household_id[
            members
        ] = current_household

        household_size[
            members
        ] = size

        cursor += size

        current_household += 1

    assert (
        household_id >= 0
    ).all()

    assert (
        household_size > 0
    ).all()

    result[
        "household_id"
    ] = household_id

    result[
        "household_size"
    ] = household_size

    return result

In [4]:
# ============================================================
# Empirical Dutch household-size distribution
#
# Source:
# CBS, private households by household size,
# Netherlands, 1 January 2026.
#
# CBS reports exact sizes 1, 2, 3, 4 and 5+.
# Because the simulator uses sizes 1--6, the 5+ category
# is approximated by a mixture of sizes 5 and 6 chosen so
# that the national CBS mean household size is preserved.
# ============================================================

HOUSEHOLD_SIZES = np.array(
    [1, 2, 3, 4, 5, 6],
    dtype=np.int64,
)


CBS_TOTAL_HOUSEHOLDS_2026 = 8_474_674

CBS_HOUSEHOLDS_1 = 3_412_202
CBS_HOUSEHOLDS_2 = 2_724_190
CBS_HOUSEHOLDS_3 = 979_779
CBS_HOUSEHOLDS_4 = 953_077
CBS_HOUSEHOLDS_5PLUS = 405_426


# Total population and institutional population,
# also from CBS, 1 January 2026.
CBS_TOTAL_PERSONS_2026 = 18_130_208
CBS_INSTITUTIONAL_PERSONS_2026 = 335_137

CBS_PRIVATE_PERSONS_2026 = (
    CBS_TOTAL_PERSONS_2026
    - CBS_INSTITUTIONAL_PERSONS_2026
)


# ------------------------------------------------------------
# Infer average size within the CBS 5+ category
# ------------------------------------------------------------

persons_in_sizes_1_to_4 = (
    1 * CBS_HOUSEHOLDS_1
    + 2 * CBS_HOUSEHOLDS_2
    + 3 * CBS_HOUSEHOLDS_3
    + 4 * CBS_HOUSEHOLDS_4
)

CBS_MEAN_SIZE_5PLUS = (
    (
        CBS_PRIVATE_PERSONS_2026
        - persons_in_sizes_1_to_4
    )
    / CBS_HOUSEHOLDS_5PLUS
)


# Approximate 5+ using only size 5 and size 6.
#
# If q is the fraction assigned size 6:
#
#     5 * (1-q) + 6*q = mean_5plus
#
# therefore:
#
#     q = mean_5plus - 5
#
CBS_5PLUS_FRACTION_SIZE_6 = (
    CBS_MEAN_SIZE_5PLUS - 5.0
)

CBS_5PLUS_FRACTION_SIZE_5 = (
    1.0
    - CBS_5PLUS_FRACTION_SIZE_6
)


p_1 = (
    CBS_HOUSEHOLDS_1
    / CBS_TOTAL_HOUSEHOLDS_2026
)

p_2 = (
    CBS_HOUSEHOLDS_2
    / CBS_TOTAL_HOUSEHOLDS_2026
)

p_3 = (
    CBS_HOUSEHOLDS_3
    / CBS_TOTAL_HOUSEHOLDS_2026
)

p_4 = (
    CBS_HOUSEHOLDS_4
    / CBS_TOTAL_HOUSEHOLDS_2026
)

p_5plus = (
    CBS_HOUSEHOLDS_5PLUS
    / CBS_TOTAL_HOUSEHOLDS_2026
)

p_5 = (
    p_5plus
    * CBS_5PLUS_FRACTION_SIZE_5
)

p_6 = (
    p_5plus
    * CBS_5PLUS_FRACTION_SIZE_6
)


HOUSEHOLD_SIZE_PROBS = np.array(
    [
        p_1,
        p_2,
        p_3,
        p_4,
        p_5,
        p_6,
    ],
    dtype=np.float64,
)


assert np.isclose(
    HOUSEHOLD_SIZE_PROBS.sum(),
    1.0,
)


print(
    "CBS household probabilities:",
    HOUSEHOLD_SIZE_PROBS,
)

print(
    "CBS mean household size:",
    np.sum(
        HOUSEHOLD_SIZES
        * HOUSEHOLD_SIZE_PROBS
    ),
)

print(
    "CBS inferred mean size of 5+ households:",
    CBS_MEAN_SIZE_5PLUS,
)

CBS household probabilities: [0.40263519 0.32145071 0.11561259 0.11246179 0.02946568 0.01837404]
CBS mean household size: 2.09979416317371
CBS inferred mean size of 5+ households: 5.384075022322199


In [5]:
def assign_age_aware_households(
    people,
    *,
    seed,
):
    """
    Reassign people to the already generated household sizes
    using age-aware rules.

    Main principles:
    - household sizes generated in Step 1a are preserved exactly;
    - CBS size-specific child-household probabilities determine
      which multi-person households are treated as family households;
    - 2-person family households may contain 1 adult + 1 child;
    - larger family households reserve two adult-caregiver places;
    - children in the same household tend to have similar ages;
    - non-family households of size >= 3 contain adults only;
    - remaining 2-person households pair people of similar ages;
    - remaining people occupy single-person households.
    """

    rng = np.random.default_rng(seed)

    result = people.copy()

    ages = result[
        "age"
    ].to_numpy(
        dtype=np.int64
    )

    n_people = len(result)

    # --------------------------------------------------------
    # Preserve the household-size distribution
    # already generated by assign_households().
    # --------------------------------------------------------

    household_sizes = (
        result
        .groupby("household_id")
        .size()
        .to_numpy(
            dtype=np.int64,
            copy=True,
        )
    )

    rng.shuffle(
        household_sizes
    )

    households = [
        []
        for _ in household_sizes
    ]

    available = set(
        range(n_people)
    )

    children = set(
        np.flatnonzero(
            ages < 18
        ).tolist()
    )

    # ========================================================
    # Helper: choose a person closest to a target age
    # ========================================================

    def take_closest(
        target_age,
        *,
        min_age=None,
        max_age=None,
    ):

        candidates = [
            person
            for person in available
            if (
                min_age is None
                or ages[person] >= min_age
            )
            and (
                max_age is None
                or ages[person] <= max_age
            )
        ]

        if not candidates:
            return None

        distances = np.array(
            [
                abs(
                    ages[person]
                    - target_age
                )
                for person in candidates
            ]
        )

        min_distance = (
            distances.min()
        )

        closest = [
            candidates[i]
            for i in range(
                len(candidates)
            )
            if (
                distances[i]
                == min_distance
            )
        ]

        chosen = int(
            rng.choice(
                closest
            )
        )

        available.remove(
            chosen
        )

        children.discard(
            chosen
        )

        return chosen

    # ========================================================
    # Helper: select children with similar ages
    # ========================================================

    def take_children(
        n_children,
    ):

        selected = []

        if n_children == 0:
            return selected

        if (
            len(children)
            < n_children
        ):
            raise ValueError(
                "Not enough children available"
            )

        # First child chosen randomly.
        anchor = int(
            rng.choice(
                list(children)
            )
        )

        selected.append(
            anchor
        )

        children.remove(
            anchor
        )

        available.remove(
            anchor
        )

        # Additional children are chosen
        # close to the mean sibling age.
        while (
            len(selected)
            < n_children
        ):

            mean_age = np.mean(
                [
                    ages[i]
                    for i in selected
                ]
            )

            candidate_list = list(
                children
            )

            distances = np.array(
                [
                    abs(
                        ages[i]
                        - mean_age
                    )
                    for i in candidate_list
                ]
            )

            minimum = (
                distances.min()
            )

            closest = [
                candidate_list[i]
                for i in range(
                    len(candidate_list)
                )
                if (
                    distances[i]
                    == minimum
                )
            ]

            chosen = int(
                rng.choice(
                    closest
                )
            )

            selected.append(
                chosen
            )

            children.remove(
                chosen
            )

            available.remove(
                chosen
            )

        return selected

    # ========================================================
    # CBS probability that a household of a given size
    # contains one or more children.
    #
    # Netherlands, 1 January 2026.
    # ========================================================

    CBS_CHILD_HOUSEHOLD_PROB = {
        2: 377_558 / 2_724_190,
        3: 944_502 / 979_779,
        4: 941_770 / 953_077,

        # CBS combines all households >= 5.
        5: 400_521 / 405_426,
        6: 400_521 / 405_426,
    }

    # ========================================================
    # 1. Select family households
    # ========================================================

    family_households = []

    for household, size in enumerate(
        household_sizes
    ):

        size = int(size)

        if size < 2:
            continue

        child_probability = (
            CBS_CHILD_HOUSEHOLD_PROB[
                size
            ]
        )

        if (
            rng.random()
            < child_probability
        ):

            family_households.append(
                household
            )

    family_household_set = set(
        family_households
    )

    # ========================================================
    # Child capacity
    #
    # size 2:
    #     1 adult + 1 child
    #
    # size >= 3:
    #     reserve two places for adult caregivers
    # ========================================================

    child_capacity = {}

    for household in family_households:

        size = int(
            household_sizes[
                household
            ]
        )

        if size == 2:

            child_capacity[
                household
            ] = 1

        else:

            child_capacity[
                household
            ] = (
                size - 2
            )

    total_child_capacity = sum(
        child_capacity.values()
    )

    if (
        len(children)
        > total_child_capacity
    ):

        raise ValueError(
            "Not enough capacity in selected "
            "family households for all children"
        )

    # Number of children placed in each
    # family household.
    child_counts = {
        household: 0
        for household
        in family_households
    }

    n_children = len(
        children
    )

    # Give one child first to as many selected
    # family households as possible.
    shuffled_family_households = (
        family_households.copy()
    )

    rng.shuffle(
        shuffled_family_households
    )

    first_round = min(
        n_children,
        len(
            shuffled_family_households
        ),
    )

    for household in (
        shuffled_family_households[
            :first_round
        ]
    ):

        child_counts[
            household
        ] += 1

    remaining_children = (
        n_children
        - first_round
    )

    # Allocate remaining children subject
    # to each household's capacity.
    while remaining_children > 0:

        eligible = [
            household
            for household
            in family_households
            if (
                child_counts[
                    household
                ]
                <
                child_capacity[
                    household
                ]
            )
        ]

        if not eligible:

            raise ValueError(
                "No family household has remaining "
                "capacity for unassigned children"
            )

        chosen_household = int(
            rng.choice(
                eligible
            )
        )

        child_counts[
            chosen_household
        ] += 1

        remaining_children -= 1

    # ========================================================
    # Fill family households
    # ========================================================

    for household in (
        family_households
    ):

        number_of_children = (
            child_counts[
                household
            ]
        )

        household_children = (
            take_children(
                number_of_children
            )
        )

        households[
            household
        ].extend(
            household_children
        )

        # ----------------------------------------------------
        # Parent target age
        # ----------------------------------------------------

        if household_children:

            child_mean_age = float(
                np.mean(
                    [
                        ages[i]
                        for i
                        in household_children
                    ]
                )
            )

            target_parent_age = int(
                np.clip(
                    child_mean_age + 30,
                    25,
                    60,
                )
            )

        else:

            target_parent_age = 40

        # ----------------------------------------------------
        # First adult caregiver
        # ----------------------------------------------------

        parent_1 = take_closest(
            target_parent_age,
            min_age=25,
            max_age=64,
        )

        if parent_1 is None:

            parent_1 = take_closest(
                target_parent_age,
                min_age=18,
            )

        if parent_1 is None:

            raise ValueError(
                "No adult available for a "
                "family household"
            )

        households[
            household
        ].append(
            parent_1
        )

        # ----------------------------------------------------
        # Second adult caregiver if household has room
        # ----------------------------------------------------

        if (
            len(
                households[
                    household
                ]
            )
            <
            household_sizes[
                household
            ]
        ):

            parent_2 = take_closest(
                ages[parent_1],
                min_age=25,
                max_age=64,
            )

            if parent_2 is None:

                parent_2 = take_closest(
                    ages[parent_1],
                    min_age=18,
                )

            if parent_2 is None:

                raise ValueError(
                    "No second adult available "
                    "for a family household"
                )

            households[
                household
            ].append(
                parent_2
            )

        # ----------------------------------------------------
        # Fill any remaining positions
        #
        # These can represent young adults,
        # grandparents, or other adult members.
        # ----------------------------------------------------

        while (
            len(
                households[
                    household
                ]
            )
            <
            household_sizes[
                household
            ]
        ):

            if (
                rng.random()
                < 0.65
            ):

                extra_person = (
                    take_closest(
                        target_parent_age
                        - 15,
                        min_age=18,
                        max_age=30,
                    )
                )

            else:

                extra_person = (
                    take_closest(
                        target_parent_age
                        + 25,
                        min_age=55,
                    )
                )

            if extra_person is None:

                extra_person = (
                    take_closest(
                        target_parent_age,
                        min_age=18,
                    )
                )

            if extra_person is None:

                raise ValueError(
                    "No person available to complete "
                    "a family household"
                )

            households[
                household
            ].append(
                extra_person
            )

    # Every child should now belong
    # to a selected family household.
    assert len(children) == 0

    # ========================================================
    # 2. Non-family households of size >= 3
    # ========================================================

    non_family_large_households = [
        i
        for i, size
        in enumerate(
            household_sizes
        )
        if (
            size >= 3
            and i
            not in family_household_set
        )
    ]

    rng.shuffle(
        non_family_large_households
    )

    for household in (
        non_family_large_households
    ):

        target_size = int(
            household_sizes[
                household
            ]
        )

        remaining_people = list(
            available
        )

        if not remaining_people:

            raise ValueError(
                "No people remain for a "
                "non-family household"
            )

        # Prefer an adult as the anchor.
        adult_candidates = [
            person
            for person in remaining_people
            if ages[person] >= 18
        ]

        if adult_candidates:

            anchor = int(
                rng.choice(
                    adult_candidates
                )
            )

        else:

            anchor = int(
                rng.choice(
                    remaining_people
                )
            )

        available.remove(
            anchor
        )

        children.discard(
            anchor
        )

        households[
            household
        ].append(
            anchor
        )

        # Fill remaining places with people
        # approximately similar in age.
        while (
            len(
                households[
                    household
                ]
            )
            < target_size
        ):

            member = take_closest(
                ages[anchor],
                min_age=18,
            )

            if member is None:

                member = take_closest(
                    ages[anchor]
                )

            if member is None:

                raise ValueError(
                    "No person available to complete "
                    "a non-family household"
                )

            households[
                household
            ].append(
                member
            )

    # ========================================================
    # 3. Non-family two-person households
    # ========================================================

    two_person_households = [
        i
        for i, size
        in enumerate(
            household_sizes
        )
        if (
            size == 2
            and i
            not in family_household_set
        )
    ]

    rng.shuffle(
        two_person_households
    )

    for household in (
        two_person_households
    ):

        remaining_people = list(
            available
        )

        if not remaining_people:

            raise ValueError(
                "No people remain for a "
                "two-person household"
            )

        # Prefer an adult anchor because all
        # children have already been allocated.
        adult_candidates = [
            person
            for person in remaining_people
            if ages[person] >= 18
        ]

        if adult_candidates:

            anchor = int(
                rng.choice(
                    adult_candidates
                )
            )

        else:

            anchor = int(
                rng.choice(
                    remaining_people
                )
            )

        available.remove(
            anchor
        )

        children.discard(
            anchor
        )

        households[
            household
        ].append(
            anchor
        )

        # Partner with someone of approximately
        # the same age.
        partner = take_closest(
            ages[anchor]
        )

        if partner is None:

            raise ValueError(
                "No partner available for "
                "a two-person household"
            )

        households[
            household
        ].append(
            partner
        )

    # ========================================================
    # 4. Single-person households
    # ========================================================

    single_households = [
        i
        for i, size
        in enumerate(
            household_sizes
        )
        if size == 1
    ]

    remaining_people = list(
        available
    )

    rng.shuffle(
        remaining_people
    )

    if (
        len(remaining_people)
        != len(single_households)
    ):

        raise ValueError(
            "Remaining people do not match "
            "the number of single households"
        )

    for household, person in zip(
        single_households,
        remaining_people,
    ):

        households[
            household
        ].append(
            int(person)
        )

        available.remove(
            int(person)
        )

    assert len(available) == 0

    # ========================================================
    # Write household assignments back to dataframe
    # ========================================================

    household_id = np.full(
        n_people,
        -1,
        dtype=np.int64,
    )

    household_size = np.zeros(
        n_people,
        dtype=np.int64,
    )

    household_type = np.empty(
        n_people,
        dtype=object,
    )

    for household, members in enumerate(
        households
    ):

        if (
            len(members)
            != household_sizes[
                household
            ]
        ):

            raise ValueError(
                "Generated household does not match "
                "its target household size"
            )

        member_ages = ages[
            members
        ]

        has_child = np.any(
            member_ages < 18
        )

        has_senior = np.any(
            member_ages >= 65
        )

        size = len(
            members
        )

        if size == 1:

            type_name = "single"

        elif (
            has_child
            and has_senior
        ):

            type_name = (
                "multigenerational"
            )

        elif has_child:

            type_name = (
                "family_with_children"
            )

        elif np.all(
            member_ages >= 65
        ):

            type_name = (
                "senior_household"
            )

        else:

            type_name = (
                "adult_household"
            )

        for person in members:

            household_id[
                person
            ] = household

            household_size[
                person
            ] = size

            household_type[
                person
            ] = type_name

    # --------------------------------------------------------
    # Final consistency checks
    # --------------------------------------------------------

    assert (
        household_id >= 0
    ).all()

    assert (
        household_size > 0
    ).all()

    result[
        "household_id"
    ] = household_id

    result[
        "household_size"
    ] = household_size

    result[
        "household_type"
    ] = household_type

    return result

In [6]:
# ============================================================
# REBUILD POPULATION + HOUSEHOLDS FROM SCRATCH
# using CBS household-size probabilities
# ============================================================

# Reload original synthetic population so that we do not
# accidentally preserve old household assignments.
people = pd.read_csv(
    "../data/synthetic_data.csv"
)

people = (
    people
    .sample(
        n=N_NODES,
        random_state=DATA_SEED,
    )
    .reset_index(drop=True)
)

# Remove an existing node_id if the source file happens
# to contain one.
if "node_id" in people.columns:
    people = people.drop(
        columns=["node_id"]
    )

people.insert(
    0,
    "node_id",
    np.arange(
        N_NODES,
        dtype=np.int64,
    ),
)

# ------------------------------------------------------------
# Sanity check: make sure we are actually using CBS
# probabilities before generating households.
# ------------------------------------------------------------

print(
    "HOUSEHOLD_SIZE_PROBS:",
    HOUSEHOLD_SIZE_PROBS,
)

print(
    "Probability sum:",
    HOUSEHOLD_SIZE_PROBS.sum(),
)

print(
    "Target mean household size:",
    np.sum(
        HOUSEHOLD_SIZES
        * HOUSEHOLD_SIZE_PROBS
    ),
)

assert np.isclose(
    HOUSEHOLD_SIZE_PROBS.sum(),
    1.0,
)

assert np.isclose(
    np.sum(
        HOUSEHOLD_SIZES
        * HOUSEHOLD_SIZE_PROBS
    ),
    2.09979416317371,
    atol=1e-6,
)

# ------------------------------------------------------------
# Generate NEW household sizes from the CBS distribution
# ------------------------------------------------------------

people = assign_households(
    people,
    seed=HOUSEHOLD_SEED,
)

# Save infection-rate vector again because people has
# just been rebuilt.
infection_rate = people[
    "infection_rate"
].to_numpy(
    dtype=np.float64
)

# ------------------------------------------------------------
# Check household sizes BEFORE age-aware reassignment
# ------------------------------------------------------------

raw_household_sizes = (
    people
    .groupby("household_id")
    .size()
)

print(
    "\nBEFORE age-aware reassignment"
)

print(
    "Number of households:",
    len(raw_household_sizes),
)

print(
    "Mean household size:",
    raw_household_sizes.mean(),
)

print(
    "\nHouseholds by size:"
)

print(
    raw_household_sizes
    .value_counts()
    .sort_index()
)

HOUSEHOLD_SIZE_PROBS: [0.40263519 0.32145071 0.11561259 0.11246179 0.02946568 0.01837404]
Probability sum: 1.0
Target mean household size: 2.09979416317371

BEFORE age-aware reassignment
Number of households: 4790
Mean household size: 2.0876826722338206

Households by size:
1    1942
2    1545
3     545
4     539
5     137
6      82
Name: count, dtype: int64


In [7]:
people = (
    assign_age_aware_households(
        people,
        seed=HOUSEHOLD_SEED,
    )
)

people.head()

,node_id,person_id,age,sex,occupation,smoking,alcohol,obesity,infection_rate,household_id,household_size,household_type
0,0,35220,62,Female,Student,0,0,0,0.3329,3789,3,adult_household
1,1,29487,71,Female,Office-based,0,0,0,0.3428,669,2,senior_household
2,2,38566,40,Male,Agriculture,1,1,1,0.3998,3462,4,family_with_children
3,3,20529,70,Male,Office-based,1,1,0,0.4563,2074,1,single
4,4,30182,54,Male,Healthcare,0,0,0,0.3793,4523,2,adult_household


In [8]:
household_summary = (
    people[
        [
            "household_id",
            "household_size",
        ]
    ]
    .drop_duplicates(
        "household_id"
    )
    ["household_size"]
    .value_counts()
    .sort_index()
)

print(
    "Number of people:",
    len(people),
)

print(
    "Number of households:",
    people[
        "household_id"
    ].nunique(),
)

print(
    "Mean household size:",
    len(people)
    / people[
        "household_id"
    ].nunique(),
)

print(
    "\nHouseholds by size:"
)

print(
    household_summary
)

print(
    "\nPeople by household size:"
)

print(
    people[
        "household_size"
    ]
    .value_counts()
    .sort_index()
)

Number of people: 10000
Number of households: 4790
Mean household size: 2.0876826722338206

Households by size:
household_size
1    1942
2    1545
3     545
4     539
5     137
6      82
Name: count, dtype: int64

People by household size:
household_size
1    1942
2    3090
3    1635
4    2156
5     685
6     492
Name: count, dtype: int64


In [9]:
actual_household_sizes = (
    people
    .groupby(
        "household_id"
    )
    .size()
)

stored_household_sizes = (
    people
    .groupby(
        "household_id"
    )[
        "household_size"
    ]
    .first()
)

assert np.array_equal(
    actual_household_sizes.to_numpy(),
    stored_household_sizes.to_numpy(),
)

assert (
    people[
        "household_id"
    ] >= 0
).all()

print(
    "Household assignment checks passed."
)

Household assignment checks passed.


In [10]:
print(
    people["age"].describe()
)

age_groups = pd.cut(
    people["age"],
    bins=[
        0,
        17,
        24,
        34,
        44,
        54,
        64,
        74,
        200,
    ],
    labels=[
        "0-17",
        "18-24",
        "25-34",
        "35-44",
        "45-54",
        "55-64",
        "65-74",
        "75+",
    ],
)

print(
    "\nPeople by age group:"
)

print(
    age_groups
    .value_counts()
    .sort_index()
)

print(
    "\nAge by occupation:"
)

print(
    people.groupby(
        "occupation"
    )["age"].agg(
        [
            "count",
            "min",
            "median",
            "max",
        ]
    )
)

count    10000.000000
mean        37.014500
std         21.292126
min          1.000000
25%         18.000000
50%         37.000000
75%         56.000000
max         74.000000
Name: age, dtype: float64

People by age group:
age
0-17     2362
18-24     943
25-34    1387
35-44    1355
45-54    1311
55-64    1365
65-74    1277
75+         0
Name: count, dtype: int64

Age by occupation:
                    count  min  median  max
occupation                                 
Agriculture          1001    1    37.0   74
Healthcare           2003    1    36.0   74
Office-based         1481    1    37.0   74
Other                1524    1    36.0   74
Sales and services   1528    1    38.0   74
Student              2463    1    37.0   74


In [11]:
# ============================================================
# CBS household-size calibration diagnostics
# ============================================================

realised_household_sizes = (
    people
    .groupby("household_id")
    .size()
)

realised_household_counts = (
    realised_household_sizes
    .value_counts()
    .sort_index()
)

realised_household_probs = (
    realised_household_counts
    / realised_household_counts.sum()
)

household_comparison = pd.DataFrame(
    {
        "size": HOUSEHOLD_SIZES,

        "CBS_target_probability":
            HOUSEHOLD_SIZE_PROBS,

        "realised_probability":
            [
                realised_household_probs.get(
                    size,
                    0.0,
                )
                for size
                in HOUSEHOLD_SIZES
            ],
    }
)

household_comparison[
    "difference"
] = (
    household_comparison[
        "realised_probability"
    ]
    - household_comparison[
        "CBS_target_probability"
    ]
)

display(
    household_comparison
)

print(
    "Number of households:",
    realised_household_sizes.size,
)

print(
    "Realised mean household size:",
    realised_household_sizes.mean(),
)

print(
    "CBS target mean household size:",
    np.sum(
        HOUSEHOLD_SIZES
        * HOUSEHOLD_SIZE_PROBS
    ),
)

print(
    "\nHouseholds by type:"
)

print(
    people
    .groupby("household_id")[
        "household_type"
    ]
    .first()
    .value_counts()
)

,size,CBS_target_probability,realised_probability,difference
0,1,0.402635,0.405428,0.002793
1,2,0.321451,0.322547,0.001096
2,3,0.115613,0.113779,-0.001834
3,4,0.112462,0.112526,0.000064
4,5,0.029466,0.028601,-0.000864
5,6,0.018374,0.017119,-0.001255


Number of households: 4790
Realised mean household size: 2.0876826722338206
CBS target mean household size: 2.09979416317371

Households by type:
household_type
single                  1942
family_with_children    1474
adult_household          978
senior_household         366
multigenerational         30
Name: count, dtype: int64


In [12]:
# ============================================================
# Final CBS household calibration check
# ============================================================

final_sizes = (
    people
    .groupby("household_id")
    .size()
)

final_counts = (
    final_sizes
    .value_counts()
    .sort_index()
)

final_probs = (
    final_counts
    / final_counts.sum()
)

comparison = pd.DataFrame(
    {
        "household_size":
            HOUSEHOLD_SIZES,

        "CBS_target":
            HOUSEHOLD_SIZE_PROBS,

        "realised":
            [
                final_probs.get(
                    size,
                    0.0,
                )
                for size
                in HOUSEHOLD_SIZES
            ],
    }
)

comparison[
    "difference"
] = (
    comparison["realised"]
    - comparison["CBS_target"]
)

comparison[
    "absolute_difference"
] = (
    comparison["difference"].abs()
)

display(comparison)

print(
    "People:",
    final_sizes.sum()
)

print(
    "Households:",
    len(final_sizes)
)

print(
    "Realised mean size:",
    final_sizes.mean()
)

print(
    "CBS target mean size:",
    np.sum(
        HOUSEHOLD_SIZES
        * HOUSEHOLD_SIZE_PROBS
    )
)

,household_size,CBS_target,realised,difference,absolute_difference
0,1,0.402635,0.405428,0.002793,0.002793
1,2,0.321451,0.322547,0.001096,0.001096
2,3,0.115613,0.113779,-0.001834,0.001834
3,4,0.112462,0.112526,0.000064,0.000064
4,5,0.029466,0.028601,-0.000864,0.000864
5,6,0.018374,0.017119,-0.001255,0.001255


People: 10000
Households: 4790
Realised mean size: 2.0876826722338206
CBS target mean size: 2.09979416317371


In [13]:
print(
    "People:",
    len(people),
)

print(
    "Households:",
    people[
        "household_id"
    ].nunique(),
)

print(
    "\nHouseholds by type:"
)

print(
    people[
        [
            "household_id",
            "household_type",
        ]
    ]
    .drop_duplicates(
        "household_id"
    )
    ["household_type"]
    .value_counts()
)

print(
    "\nHouseholds by size:"
)

print(
    people[
        [
            "household_id",
            "household_size",
        ]
    ]
    .drop_duplicates(
        "household_id"
    )
    ["household_size"]
    .value_counts()
    .sort_index()
)

People: 10000
Households: 4790

Households by type:
household_type
single                  1942
family_with_children    1474
adult_household          978
senior_household         366
multigenerational         30
Name: count, dtype: int64

Households by size:
household_size
1    1942
2    1545
3     545
4     539
5     137
6      82
Name: count, dtype: int64


In [14]:
bad_child_households = []

for household_id, group in (
    people.groupby(
        "household_id"
    )
):

    ages = group[
        "age"
    ].to_numpy()

    children = ages[
        ages < 18
    ]

    adults = ages[
        ages >= 18
    ]

    if len(children) > 0:

        if len(adults) == 0:

            bad_child_households.append(
                household_id
            )

print(
    "Child households without adults:",
    len(
        bad_child_households
    ),
)

Child households without adults: 0


## 3. Contact contexts and primary layered network


In [15]:
CONTACT_CONTEXT_SEED = 20261021


def add_contact_context(
    people,
    *,
    seed,
):
    rng = np.random.default_rng(seed)

    result = people.copy()

    context = np.empty(
        len(result),
        dtype=object,
    )

    ages = result[
        "age"
    ].to_numpy()

    for i, age in enumerate(ages):

        if age <= 17:
            context[i] = "school"

        elif age <= 24:
            context[i] = rng.choice(
                [
                    "university",
                    "workplace",
                    "other",
                ],
                p=[
                    0.55,
                    0.35,
                    0.10,
                ],
            )

        elif age <= 64:
            context[i] = rng.choice(
                [
                    "workplace",
                    "other",
                ],
                p=[
                    0.90,
                    0.10,
                ],
            )

        else:
            context[i] = rng.choice(
                [
                    "retired",
                    "other",
                ],
                p=[
                    0.90,
                    0.10,
                ],
            )

    result[
        "contact_context"
    ] = context

    return result


people = add_contact_context(
    people,
    seed=CONTACT_CONTEXT_SEED,
)

In [16]:
NETWORK_SEED = 22

TARGET_MEAN_DEGREE = 15.446

rng = np.random.default_rng(
    NETWORK_SEED
)


# ============================================================
# Helper
# ============================================================

def normalise_edge(
    u,
    v,
):
    u = int(u)
    v = int(v)

    if u == v:
        return None

    return (
        min(u, v),
        max(u, v),
    )


# ============================================================
# 1. Household edges
# ============================================================

household_edges = set()

for _, group in people.groupby(
    "household_id"
):

    members = (
        group["node_id"]
        .to_numpy(
            dtype=np.int64
        )
    )

    for i in range(
        len(members)
    ):

        for j in range(
            i + 1,
            len(members),
        ):

            household_edges.add(
                (
                    int(members[i]),
                    int(members[j]),
                )
            )


print(
    "Household edges:",
    len(household_edges),
)


# ============================================================
# 2. Assign context groups
# ============================================================

context_group_id = np.full(
    N_NODES,
    -1,
    dtype=np.int64,
)

next_group_id = 0


def assign_groups(
    node_ids,
    *,
    approximate_group_size,
):
    global next_group_id

    node_ids = np.array(
        node_ids,
        dtype=np.int64,
        copy=True,
    )

    rng.shuffle(
        node_ids
    )

    groups = []

    for start in range(
        0,
        len(node_ids),
        approximate_group_size,
    ):

        group = node_ids[
            start:
            start
            + approximate_group_size
        ]

        if len(group) == 0:
            continue

        group_id = next_group_id
        next_group_id += 1

        context_group_id[
            group
        ] = group_id

        groups.append(
            group
        )

    return groups


school_nodes = people.loc[
    people["contact_context"]
    == "school",
    "node_id",
].to_numpy()

university_nodes = people.loc[
    people["contact_context"]
    == "university",
    "node_id",
].to_numpy()

workplace_nodes = people.loc[
    people["contact_context"]
    == "workplace",
    "node_id",
].to_numpy()


school_groups = assign_groups(
    school_nodes,
    approximate_group_size=20,
)

university_groups = assign_groups(
    university_nodes,
    approximate_group_size=18,
)

workplace_groups = assign_groups(
    workplace_nodes,
    approximate_group_size=12,
)


people[
    "context_group_id"
] = context_group_id


print(
    "School groups:",
    len(school_groups),
)

print(
    "University groups:",
    len(university_groups),
)

print(
    "Workplace groups:",
    len(workplace_groups),
)


# ============================================================
# 3. Context edges
# ============================================================

context_edges = set()


def add_sparse_group_edges(
    groups,
    *,
    target_degree,
):

    for group in groups:

        n = len(group)

        if n < 2:
            continue

        probability = min(
            1.0,
            target_degree
            / (n - 1),
        )

        for i in range(n):

            for j in range(
                i + 1,
                n,
            ):

                if (
                    rng.random()
                    < probability
                ):

                    edge = normalise_edge(
                        group[i],
                        group[j],
                    )

                    if (
                        edge is not None
                        and edge
                        not in household_edges
                    ):
                        context_edges.add(
                            edge
                        )


add_sparse_group_edges(
    school_groups,
    target_degree=8,
)

add_sparse_group_edges(
    university_groups,
    target_degree=6,
)

add_sparse_group_edges(
    workplace_groups,
    target_degree=5,
)


print(
    "Context edges:",
    len(context_edges),
)


# ============================================================
# 4. Community edges
# ============================================================

all_edges = (
    household_edges
    | context_edges
)

target_n_edges = int(
    round(
        N_NODES
        * TARGET_MEAN_DEGREE
        / 2
    )
)

community_edges = set()

ages = people[
    "age"
].to_numpy(
    dtype=np.int64
)

households = people[
    "household_id"
].to_numpy(
    dtype=np.int64
)


max_attempts = (
    target_n_edges
    * 100
)

attempts = 0


while (
    len(all_edges)
    < target_n_edges
    and attempts
    < max_attempts
):

    attempts += 1

    u, v = rng.choice(
        N_NODES,
        size=2,
        replace=False,
    )

    u = int(u)
    v = int(v)

    # Household pairs are already represented
    # explicitly by household edges.
    if (
        households[u]
        == households[v]
    ):
        continue

    edge = normalise_edge(
        u,
        v,
    )

    if edge in all_edges:
        continue

    # Mild age assortativity:
    # people closer in age are somewhat
    # more likely to form community contacts.
    age_difference = abs(
        int(ages[u])
        - int(ages[v])
    )

    acceptance_probability = (
        np.exp(
            -age_difference
            / 25.0
        )
    )

    if (
        rng.random()
        > acceptance_probability
    ):
        continue

    community_edges.add(
        edge
    )

    all_edges.add(
        edge
    )


if (
    len(all_edges)
    < target_n_edges
):
    raise RuntimeError(
        "Could not generate enough "
        "community edges."
    )


# ============================================================
# 5. Build graph
# ============================================================

layered_edges = np.asarray(
    sorted(
        all_edges
    ),
    dtype=np.int64,
)

graph = Graph.from_edges(
    N_NODES,
    layered_edges,
)


print(
    "\nLayered network"
)

print(
    "Nodes:",
    graph.n_nodes,
)

print(
    "Edges:",
    graph.n_edges,
)

print(
    "Mean degree:",
    graph.degree().mean(),
)

print(
    "Min degree:",
    graph.degree().min(),
)

print(
    "Max degree:",
    graph.degree().max(),
)

print(
    "\nEdge layers:"
)

print(
    "Household:",
    len(household_edges),
)

print(
    "Context:",
    len(context_edges),
)

print(
    "Community:",
    len(community_edges),
)

Household edges: 9014
School groups: 119
University groups: 29
Workplace groups: 436
Context edges: 24154

Layered network
Nodes: 10000
Edges: 77230
Mean degree: 15.446
Min degree: 0
Max degree: 34

Edge layers:
Household: 9014
Context: 24154
Community: 44062


In [17]:
HOUSEHOLD_TRANSMISSION_MULTIPLIER = 2.0

household_ids = people[
    "household_id"
].to_numpy(
    dtype=np.int64
)

# One multiplier for every directed CSR entry.
# graph.indices contains every undirected edge twice:
# u -> v and v -> u.
edge_multipliers = np.ones(
    graph.indices.size,
    dtype=np.float64,
)

for node in range(
    graph.n_nodes
):

    start = graph.indptr[node]
    stop = graph.indptr[node + 1]

    neighbours = graph.indices[
        start:stop
    ]

    same_household = (
        household_ids[neighbours]
        == household_ids[node]
    )

    edge_multipliers[
        start:stop
    ][same_household] = (
        HOUSEHOLD_TRANSMISSION_MULTIPLIER
    )


print(
    "Directed graph entries:",
    len(edge_multipliers),
)

print(
    "Household-directed entries:",
    np.count_nonzero(
        edge_multipliers
        == HOUSEHOLD_TRANSMISSION_MULTIPLIER
    ),
)

print(
    "Expected household-directed entries:",
    2 * len(household_edges),
)

print(
    "Ordinary-directed entries:",
    np.count_nonzero(
        edge_multipliers == 1.0
    ),
)

Directed graph entries: 154460
Household-directed entries: 18028
Expected household-directed entries: 18028
Ordinary-directed entries: 136432


In [18]:
assert np.count_nonzero(
    edge_multipliers
    == HOUSEHOLD_TRANSMISSION_MULTIPLIER
) == 2 * len(
    household_edges
)

assert np.all(
    edge_multipliers >= 1.0
)

print(
    "Edge multiplier checks passed."
)

Edge multiplier checks passed.


## 4. Transmission-probability calibration


In [19]:
# ============================================================
# Calibrate global transmission probability p
#
# Target:
#     mean control attack rate = 40%
#
# Calibration is performed on the primary network
# (NETWORK_SEED = 22) using the heterogeneous
# susceptibility values from infection_rate.
#
# The same initial infected sets and simulator RNG seeds
# are reused for every candidate p to reduce Monte Carlo noise.
# ============================================================

RECOVERY_PROB = 0.4

TARGET_ATTACK_RATE = 0.40

P_CALIBRATION_RUNS = 30
P_CALIBRATION_N_SEEDS = 50
P_CALIBRATION_SEED = 20261022

P_LOW = 0.08
P_HIGH = 0.10

P_BISECTION_ITERATIONS = 7


# ------------------------------------------------------------
# Control parameters used for calibration
# ------------------------------------------------------------

calibration_params = NodeParams(
    susceptibility=infection_rate,
    infectiousness=np.ones(
        N_NODES,
        dtype=np.float64,
    ),
    recovery=np.full(
        N_NODES,
        RECOVERY_PROB,
        dtype=np.float64,
    ),
)


# ------------------------------------------------------------
# Matched calibration design
# ------------------------------------------------------------

calibration_rng = np.random.default_rng(
    P_CALIBRATION_SEED
)

calibration_seed_sets = [
    np.sort(
        calibration_rng.choice(
            N_NODES,
            size=P_CALIBRATION_N_SEEDS,
            replace=False,
        )
    )
    for _ in range(
        P_CALIBRATION_RUNS
    )
]

calibration_run_rng_seeds = (
    calibration_rng.integers(
        0,
        np.iinfo(np.int64).max,
        size=P_CALIBRATION_RUNS,
        dtype=np.int64,
    )
)


# ------------------------------------------------------------
# Evaluate one candidate p
# ------------------------------------------------------------

def evaluate_calibration_p(
    p_value,
):

    runs = [
        simulate(
            graph,
            seeds=seeds,
            p=float(p_value),
            params=calibration_params,
            edge_multipliers=edge_multipliers,
            rng=int(run_seed),
        )
        for seeds, run_seed
        in zip(
            calibration_seed_sets,
            calibration_run_rng_seeds,
        )
    ]

    attack_rates = np.array(
        [
            run.final_size
            / N_NODES
            for run in runs
        ],
        dtype=np.float64,
    )

    return attack_rates


# ------------------------------------------------------------
# Check initial bracket
# ------------------------------------------------------------

low_attack_rates = (
    evaluate_calibration_p(
        P_LOW
    )
)

high_attack_rates = (
    evaluate_calibration_p(
        P_HIGH
    )
)


def print_calibration_summary(
    label,
    p_value,
    attack_rates,
):
    print(
        label,
        f"p={p_value:.8f}",
        f"mean={attack_rates.mean():.6f}",
        f"sd={attack_rates.std(ddof=1):.6f}",
        f"median={np.median(attack_rates):.6f}",
        f"min={attack_rates.min():.6f}",
        f"max={attack_rates.max():.6f}",
    )


print_calibration_summary(
    "LOW ",
    P_LOW,
    low_attack_rates,
)

print_calibration_summary(
    "HIGH",
    P_HIGH,
    high_attack_rates,
)


low_mean = (
    low_attack_rates.mean()
)

high_mean = (
    high_attack_rates.mean()
)


if not (
    low_mean
    <= TARGET_ATTACK_RATE
    <= high_mean
):
    raise ValueError(
        "Initial p bracket does not contain "
        "the target attack rate. "
        f"low={low_mean:.4f}, "
        f"target={TARGET_ATTACK_RATE:.4f}, "
        f"high={high_mean:.4f}"
    )


# ------------------------------------------------------------
# Bisection calibration
# ------------------------------------------------------------

p_low = float(P_LOW)
p_high = float(P_HIGH)

calibration_history = []

for iteration in range(
    1,
    P_BISECTION_ITERATIONS + 1,
):

    p_mid = (
        p_low
        + p_high
    ) / 2.0

    mid_attack_rates = (
        evaluate_calibration_p(
            p_mid
        )
    )

    mid_mean = float(
        mid_attack_rates.mean()
    )

    calibration_history.append(
        {
            "iteration": iteration,
            "p": p_mid,
            "mean_attack_rate": mid_mean,
            "sd_attack_rate": float(
                mid_attack_rates.std(
                    ddof=1
                )
            ),
        }
    )

    print(
        f"Iteration {iteration}: "
        f"p={p_mid:.8f}, "
        f"mean attack rate="
        f"{mid_mean:.6f}"
    )

    if (
        mid_mean
        < TARGET_ATTACK_RATE
    ):
        p_low = p_mid

    else:
        p_high = p_mid


# ------------------------------------------------------------
# Final calibrated value
#
# Use midpoint of final bisection interval.
# ------------------------------------------------------------

CALIBRATED_P = (
    p_low
    + p_high
) / 2.0


print(
    "\nFinal calibration interval:",
    p_low,
    p_high,
)

print(
    "CALIBRATED_P:",
    CALIBRATED_P,
)


calibration_history_df = (
    pd.DataFrame(
        calibration_history
    )
)

display(
    calibration_history_df
)

LOW  p=0.08000000 mean=0.192300 sd=0.046917 median=0.193700 min=0.071900 max=0.265200
HIGH p=0.10000000 mean=0.455157 sd=0.018880 median=0.455150 min=0.411600 max=0.496500
Iteration 1: p=0.09000000, mean attack rate=0.337220
Iteration 2: p=0.09500000, mean attack rate=0.400457
Iteration 3: p=0.09250000, mean attack rate=0.367670
Iteration 4: p=0.09375000, mean attack rate=0.384870
Iteration 5: p=0.09437500, mean attack rate=0.394313
Iteration 6: p=0.09468750, mean attack rate=0.394553
Iteration 7: p=0.09484375, mean attack rate=0.400210

Final calibration interval: 0.09468750000000001 0.09484375
CALIBRATED_P: 0.094765625


,iteration,p,mean_attack_rate,sd_attack_rate
0,1,0.090000,0.337220,0.027229
1,2,0.095000,0.400457,0.023540
2,3,0.092500,0.367670,0.023291
3,4,0.093750,0.384870,0.020399
4,5,0.094375,0.394313,0.018426
5,6,0.094688,0.394553,0.022952
6,7,0.094844,0.400210,0.020824


In [20]:
# ============================================================
# Independent validation of calibrated p
# ============================================================

P_VALIDATION_RUNS = 100
P_VALIDATION_N_SEEDS = 50
P_VALIDATION_SEED = 20261023


validation_rng = np.random.default_rng(
    P_VALIDATION_SEED
)


validation_seed_sets = [
    np.sort(
        validation_rng.choice(
            N_NODES,
            size=P_VALIDATION_N_SEEDS,
            replace=False,
        )
    )
    for _ in range(
        P_VALIDATION_RUNS
    )
]


validation_run_rng_seeds = (
    validation_rng.integers(
        0,
        np.iinfo(np.int64).max,
        size=P_VALIDATION_RUNS,
        dtype=np.int64,
    )
)


validation_runs = [
    simulate(
        graph,
        seeds=seeds,
        p=CALIBRATED_P,
        params=calibration_params,
        edge_multipliers=edge_multipliers,
        rng=int(run_seed),
    )
    for seeds, run_seed
    in zip(
        validation_seed_sets,
        validation_run_rng_seeds,
    )
]


validation_attack_rates = np.array(
    [
        run.final_size
        / N_NODES
        for run
        in validation_runs
    ],
    dtype=np.float64,
)


print(
    "Calibrated P:",
    CALIBRATED_P,
)

print(
    "Mean attack rate:",
    validation_attack_rates.mean(),
)

print(
    "SD attack rate:",
    validation_attack_rates.std(
        ddof=1
    ),
)

print(
    "Median attack rate:",
    np.median(
        validation_attack_rates
    ),
)

print(
    "Min / max:",
    validation_attack_rates.min(),
    validation_attack_rates.max(),
)

Calibrated P: 0.094765625
Mean attack rate: 0.398688
SD attack rate: 0.022551141788379497
Median attack rate: 0.40105
Min / max: 0.3248 0.4411


## 5. Shared experiment definitions and network robustness


In [21]:
# ============================================================
# Shared experiment definitions
# ============================================================

# ------------------------------------------------------------
# Planted subgroup
# ------------------------------------------------------------

planted_mask = (
    (people["age"] >= 60)
    & (people["smoking"] == 1)
).to_numpy(
    dtype=bool
)

print(
    "Planted subgroup size:",
    planted_mask.sum(),
)

print(
    "Planted subgroup fraction:",
    planted_mask.mean(),
)


# ------------------------------------------------------------
# Minor-outbreak threshold
# ------------------------------------------------------------

MIN_MAJOR_ATTACK_RATE = 0.01

MIN_MAJOR_SIZE = int(
    np.ceil(
        MIN_MAJOR_ATTACK_RATE
        * N_NODES
    )
)

print(
    "Minor outbreak threshold:",
    f"< {MIN_MAJOR_SIZE} infections",
)


def filter_major_runs(
    runs,
    *,
    label="Runs",
):

    final_sizes = np.array(
        [
            run.final_size
            for run in runs
        ],
        dtype=np.int64,
    )

    keep_mask = (
        final_sizes
        >= MIN_MAJOR_SIZE
    )

    n_minor = int(
        (~keep_mask).sum()
    )

    n_total = len(runs)

    print(
        f"{label}: "
        f"{n_minor}/{n_total} minor outbreaks "
        f"({100 * n_minor / n_total:.2f}%)"
    )

    filtered_runs = [
        run
        for run, keep
        in zip(
            runs,
            keep_mask,
        )
        if keep
    ]

    return (
        filtered_runs,
        keep_mask,
    )


def filter_major_run_pairs(
    control_runs,
    planted_runs,
    *,
    control_label="Control",
    planted_label="Planted",
):

    if len(control_runs) != len(
        planted_runs
    ):
        raise ValueError(
            "Control and planted ensembles "
            "must contain the same number of runs."
        )

    control_sizes = np.array(
        [
            run.final_size
            for run in control_runs
        ],
        dtype=np.int64,
    )

    planted_sizes = np.array(
        [
            run.final_size
            for run in planted_runs
        ],
        dtype=np.int64,
    )

    control_major = (
        control_sizes
        >= MIN_MAJOR_SIZE
    )

    planted_major = (
        planted_sizes
        >= MIN_MAJOR_SIZE
    )

    pair_keep = (
        control_major
        & planted_major
    )

    n_runs = len(control_runs)

    n_control_minor = int(
        (~control_major).sum()
    )

    n_planted_minor = int(
        (~planted_major).sum()
    )

    n_pairs_removed = int(
        (~pair_keep).sum()
    )

    print(
        f"{control_label} minor outbreaks: "
        f"{n_control_minor}/{n_runs} "
        f"({100 * n_control_minor / n_runs:.2f}%)"
    )

    print(
        f"{planted_label} minor outbreaks: "
        f"{n_planted_minor}/{n_runs} "
        f"({100 * n_planted_minor / n_runs:.2f}%)"
    )

    print(
        "Matched pairs excluded:",
        f"{n_pairs_removed}/{n_runs}",
        f"({100 * n_pairs_removed / n_runs:.2f}%)",
    )

    print(
        "Matched pairs retained:",
        int(pair_keep.sum()),
    )

    filtered_control = [
        run
        for run, keep
        in zip(
            control_runs,
            pair_keep,
        )
        if keep
    ]

    filtered_planted = [
        run
        for run, keep
        in zip(
            planted_runs,
            pair_keep,
        )
        if keep
    ]

    return (
        filtered_control,
        filtered_planted,
        pair_keep,
    )

Planted subgroup size: 1161
Planted subgroup fraction: 0.1161
Minor outbreak threshold: < 100 infections


In [22]:
# ============================================================
# Reusable layered-network generator
#
# Used later for validation across independent network
# realisations. The population and households remain fixed;
# context grouping, context edges, and community edges are
# redrawn from network_seed.
# ============================================================

def build_layered_network(
    people,
    *,
    network_seed,
    target_mean_degree=TARGET_MEAN_DEGREE,
    household_transmission_multiplier=(
        HOUSEHOLD_TRANSMISSION_MULTIPLIER
    ),
):
    rng = np.random.default_rng(
        network_seed
    )

    n_nodes = len(people)

    # --------------------------------------------------------
    # Basic checks
    # --------------------------------------------------------

    node_ids = people[
        "node_id"
    ].to_numpy(
        dtype=np.int64
    )

    if not np.array_equal(
        node_ids,
        np.arange(
            n_nodes,
            dtype=np.int64,
        ),
    ):
        raise ValueError(
            "node_id must run from 0 to N-1."
        )

    ages = people[
        "age"
    ].to_numpy(
        dtype=np.int64
    )

    households = people[
        "household_id"
    ].to_numpy(
        dtype=np.int64
    )

    # --------------------------------------------------------
    # Helper
    # --------------------------------------------------------

    def normalise_edge_local(
        u,
        v,
    ):
        u = int(u)
        v = int(v)

        if u == v:
            return None

        return (
            min(u, v),
            max(u, v),
        )

    # ========================================================
    # 1. Household edges
    #
    # Household membership is fixed across network
    # realisations.
    # ========================================================

    household_edges_local = set()

    for _, group in people.groupby(
        "household_id"
    ):

        members = (
            group["node_id"]
            .to_numpy(
                dtype=np.int64
            )
        )

        for i in range(
            len(members)
        ):
            for j in range(
                i + 1,
                len(members),
            ):
                household_edges_local.add(
                    (
                        int(members[i]),
                        int(members[j]),
                    )
                )

    # ========================================================
    # 2. Context groups
    #
    # Context categories remain fixed, but actual school,
    # university and workplace groups are redrawn.
    # ========================================================

    context_group_id_local = np.full(
        n_nodes,
        -1,
        dtype=np.int64,
    )

    next_group_id = 0

    def assign_groups_local(
        node_ids,
        *,
        approximate_group_size,
    ):
        nonlocal next_group_id

        node_ids = np.array(
            node_ids,
            dtype=np.int64,
            copy=True,
        )

        rng.shuffle(
            node_ids
        )

        groups = []

        for start in range(
            0,
            len(node_ids),
            approximate_group_size,
        ):

            group = node_ids[
                start:
                start
                + approximate_group_size
            ]

            if len(group) == 0:
                continue

            group_id = next_group_id
            next_group_id += 1

            context_group_id_local[
                group
            ] = group_id

            groups.append(
                group
            )

        return groups

    school_nodes = people.loc[
        people["contact_context"]
        == "school",
        "node_id",
    ].to_numpy()

    university_nodes = people.loc[
        people["contact_context"]
        == "university",
        "node_id",
    ].to_numpy()

    workplace_nodes = people.loc[
        people["contact_context"]
        == "workplace",
        "node_id",
    ].to_numpy()

    school_groups = (
        assign_groups_local(
            school_nodes,
            approximate_group_size=20,
        )
    )

    university_groups = (
        assign_groups_local(
            university_nodes,
            approximate_group_size=18,
        )
    )

    workplace_groups = (
        assign_groups_local(
            workplace_nodes,
            approximate_group_size=12,
        )
    )

    # ========================================================
    # 3. Context edges
    # ========================================================

    context_edges_local = set()

    def add_sparse_group_edges_local(
        groups,
        *,
        target_degree,
    ):

        for group in groups:

            n = len(group)

            if n < 2:
                continue

            probability = min(
                1.0,
                target_degree
                / (n - 1),
            )

            for i in range(n):
                for j in range(
                    i + 1,
                    n,
                ):

                    if (
                        rng.random()
                        < probability
                    ):

                        edge = (
                            normalise_edge_local(
                                group[i],
                                group[j],
                            )
                        )

                        if (
                            edge is not None
                            and edge
                            not in
                            household_edges_local
                        ):
                            context_edges_local.add(
                                edge
                            )

    add_sparse_group_edges_local(
        school_groups,
        target_degree=8,
    )

    add_sparse_group_edges_local(
        university_groups,
        target_degree=6,
    )

    add_sparse_group_edges_local(
        workplace_groups,
        target_degree=5,
    )

    # ========================================================
    # 4. Community edges
    # ========================================================

    all_edges_local = (
        household_edges_local
        | context_edges_local
    )

    target_n_edges = int(
        round(
            n_nodes
            * target_mean_degree
            / 2
        )
    )

    community_edges_local = set()

    max_attempts = (
        target_n_edges
        * 100
    )

    attempts = 0

    while (
        len(all_edges_local)
        < target_n_edges
        and attempts
        < max_attempts
    ):

        attempts += 1

        u, v = rng.choice(
            n_nodes,
            size=2,
            replace=False,
        )

        u = int(u)
        v = int(v)

        # Household pairs are already represented.
        if (
            households[u]
            == households[v]
        ):
            continue

        edge = (
            normalise_edge_local(
                u,
                v,
            )
        )

        if edge in all_edges_local:
            continue

        # Mild age assortativity.
        age_difference = abs(
            int(ages[u])
            - int(ages[v])
        )

        acceptance_probability = np.exp(
            -age_difference
            / 25.0
        )

        if (
            rng.random()
            > acceptance_probability
        ):
            continue

        community_edges_local.add(
            edge
        )

        all_edges_local.add(
            edge
        )

    if (
        len(all_edges_local)
        < target_n_edges
    ):
        raise RuntimeError(
            "Could not generate enough "
            "community edges."
        )

    # ========================================================
    # 5. Build Graph
    # ========================================================

    layered_edges_local = np.asarray(
        sorted(
            all_edges_local
        ),
        dtype=np.int64,
    )

    graph_local = Graph.from_edges(
        n_nodes,
        layered_edges_local,
    )

    # ========================================================
    # 6. Directed edge multipliers
    # ========================================================

    edge_multipliers_local = np.ones(
        graph_local.indices.size,
        dtype=np.float64,
    )

    for node in range(
        graph_local.n_nodes
    ):

        start = (
            graph_local.indptr[node]
        )

        stop = (
            graph_local.indptr[
                node + 1
            ]
        )

        neighbours = (
            graph_local.indices[
                start:stop
            ]
        )

        same_household = (
            households[neighbours]
            == households[node]
        )

        edge_multipliers_local[
            start:stop
        ][same_household] = (
            household_transmission_multiplier
        )

    # ========================================================
    # 7. Validation
    # ========================================================

    expected_household_directed = (
        2
        * len(
            household_edges_local
        )
    )

    actual_household_directed = (
        np.count_nonzero(
            edge_multipliers_local
            == household_transmission_multiplier
        )
    )

    if (
        actual_household_directed
        != expected_household_directed
    ):
        raise RuntimeError(
            "Household edge-multiplier "
            "count is inconsistent."
        )

    # ========================================================
    # 8. Summary information
    # ========================================================

    degree = (
        graph_local.degree()
    )

    info = {
        "network_seed":
            int(network_seed),

        "n_nodes":
            int(graph_local.n_nodes),

        "n_edges":
            int(graph_local.n_edges),

        "mean_degree":
            float(degree.mean()),

        "min_degree":
            int(degree.min()),

        "max_degree":
            int(degree.max()),

        "household_edges":
            len(
                household_edges_local
            ),

        "context_edges":
            len(
                context_edges_local
            ),

        "community_edges":
            len(
                community_edges_local
            ),

        "school_groups":
            len(
                school_groups
            ),

        "university_groups":
            len(
                university_groups
            ),

        "workplace_groups":
            len(
                workplace_groups
            ),
    }

    return (
        graph_local,
        edge_multipliers_local,
        info,
    )

In [23]:
# ============================================================
# Reproducibility check
#
# The reusable builder should reproduce the original graph
# exactly for NETWORK_SEED = 22.
# ============================================================

(
    test_graph_22,
    test_edge_multipliers_22,
    test_info_22,
) = build_layered_network(
    people,
    network_seed=22,
)

print(test_info_22)

assert np.array_equal(
    test_graph_22.indptr,
    graph.indptr,
)

assert np.array_equal(
    test_graph_22.indices,
    graph.indices,
)

assert np.array_equal(
    test_edge_multipliers_22,
    edge_multipliers,
)

print(
    "\nSeed 22 exactly reproduces "
    "the original network."
)

{'network_seed': 22, 'n_nodes': 10000, 'n_edges': 77230, 'mean_degree': 15.446, 'min_degree': 0, 'max_degree': 34, 'household_edges': 9014, 'context_edges': 24154, 'community_edges': 44062, 'school_groups': 119, 'university_groups': 29, 'workplace_groups': 436}

Seed 22 exactly reproduces the original network.


In [24]:
# ============================================================
# Independent network realisations
#
# Final robustness validation uses five independent network
# realisations while keeping the population and households fixed.
# ============================================================

NETWORK_REALISATION_SEEDS = [
    22,
    23,
    24,
    25,
    26,
]

network_realisations = {}

network_summary_rows = []

for network_seed in (
    NETWORK_REALISATION_SEEDS
):

    print(
        "\nBuilding network:",
        network_seed,
    )

    if network_seed == 22:
        # Reuse the network already built
        # during the reproducibility check.
        current_graph = (
            test_graph_22
        )

        current_edge_multipliers = (
            test_edge_multipliers_22
        )

        current_info = (
            test_info_22
        )

    else:
        (
            current_graph,
            current_edge_multipliers,
            current_info,
        ) = build_layered_network(
            people,
            network_seed=network_seed,
        )

    network_realisations[
        network_seed
    ] = {
        "graph":
            current_graph,

        "edge_multipliers":
            current_edge_multipliers,
    }

    network_summary_rows.append(
        current_info
    )


network_summary = pd.DataFrame(
    network_summary_rows
)

display(
    network_summary
)


Building network: 22

Building network: 23

Building network: 24

Building network: 25

Building network: 26


,network_seed,n_nodes,n_edges,mean_degree,min_degree,max_degree,household_edges,context_edges,community_edges,school_groups,university_groups,workplace_groups
0,22,10000,77230,15.446,0,34,9014,24154,44062,119,29,436
1,23,10000,77230,15.446,0,35,9014,24161,44055,119,29,436
2,24,10000,77230,15.446,1,33,9014,23916,44300,119,29,436
3,25,10000,77230,15.446,0,32,9014,24151,44065,119,29,436
4,26,10000,77230,15.446,1,32,9014,24049,44167,119,29,436


In [25]:
# ============================================================
# Check calibrated P across independent network realisations
# ============================================================

NETWORK_CHECK_RUNS = 30
NETWORK_CHECK_N_SEEDS = 50
NETWORK_CHECK_SEED = 20261024

network_check_rng = np.random.default_rng(
    NETWORK_CHECK_SEED
)

# Same initial infections and RNG seeds on every network.
# This makes differences primarily attributable to the network.
network_check_seed_sets = [
    np.sort(
        network_check_rng.choice(
            N_NODES,
            size=NETWORK_CHECK_N_SEEDS,
            replace=False,
        )
    )
    for _ in range(
        NETWORK_CHECK_RUNS
    )
]

network_check_run_rng_seeds = (
    network_check_rng.integers(
        0,
        np.iinfo(np.int64).max,
        size=NETWORK_CHECK_RUNS,
        dtype=np.int64,
    )
)


network_attack_rows = []

for network_seed in (
    NETWORK_REALISATION_SEEDS
):

    current_graph = (
        network_realisations[
            network_seed
        ]["graph"]
    )

    current_edge_multipliers = (
        network_realisations[
            network_seed
        ]["edge_multipliers"]
    )

    current_runs = [
        simulate(
            current_graph,
            seeds=seeds,
            p=CALIBRATED_P,
            params=calibration_params,
            edge_multipliers=(
                current_edge_multipliers
            ),
            rng=int(run_seed),
        )
        for seeds, run_seed
        in zip(
            network_check_seed_sets,
            network_check_run_rng_seeds,
        )
    ]

    final_sizes = np.array(
        [
            run.final_size
            for run in current_runs
        ],
        dtype=np.int64,
    )

    attack_rates = (
        final_sizes
        / N_NODES
    )

    minor_fraction = float(
        np.mean(
            final_sizes
            < MIN_MAJOR_SIZE
        )
    )

    network_attack_rows.append(
        {
            "network_seed":
                network_seed,

            "mean_attack_rate":
                attack_rates.mean(),

            "sd_attack_rate":
                attack_rates.std(
                    ddof=1
                ),

            "median_attack_rate":
                np.median(
                    attack_rates
                ),

            "min_attack_rate":
                attack_rates.min(),

            "max_attack_rate":
                attack_rates.max(),

            "minor_outbreak_fraction":
                minor_fraction,
        }
    )


network_attack_summary = (
    pd.DataFrame(
        network_attack_rows
    )
)

display(
    network_attack_summary
)

print(
    "\nMean across network means:",
    network_attack_summary[
        "mean_attack_rate"
    ].mean(),
)

print(
    "Range of network means:",
    (
        network_attack_summary[
            "mean_attack_rate"
        ].min(),
        network_attack_summary[
            "mean_attack_rate"
        ].max(),
    ),
)

,network_seed,mean_attack_rate,sd_attack_rate,median_attack_rate,min_attack_rate,max_attack_rate,minor_outbreak_fraction
0,22,0.399227,0.025012,0.39230,0.3575,0.4604,0.0
1,23,0.398400,0.019158,0.39925,0.3562,0.4317,0.0
2,24,0.398563,0.018355,0.40050,0.3646,0.4349,0.0
3,25,0.396153,0.021988,0.39540,0.3550,0.4393,0.0
4,26,0.393683,0.020459,0.39385,0.3530,0.4384,0.0



Mean across network means: 0.3972053333333333
Range of network means: (np.float64(0.3936833333333334), np.float64(0.3992266666666666))


## 6. Model B cross-network validation


In [26]:
# ============================================================
# Model B validation across independent network realisations
# ============================================================

NETWORK_MODEL_B_RUNS = 100
NETWORK_MODEL_B_N_SEEDS = 50

NETWORK_MODEL_B_MULTIPLIER = 1.75

NETWORK_MODEL_B_DESIGN_SEED = 20261025

NETWORK_MODEL_B_TARGET = (
    "smoking == 1 AND age >= 60"
)

NETWORK_MODEL_B_CATEGORICAL_COLUMNS = [
    "sex",
    "occupation",
    "smoking",
    "alcohol",
    "obesity",
]


# ------------------------------------------------------------
# Fresh Model B parameters
# ------------------------------------------------------------

network_model_b_base_params = NodeParams(
    susceptibility=infection_rate,
    infectiousness=np.ones(
        N_NODES,
        dtype=np.float64,
    ),
    recovery=np.full(
        N_NODES,
        RECOVERY_PROB,
        dtype=np.float64,
    ),
)

network_model_b_planted_params = (
    network_model_b_base_params.scaled(
        planted_mask,
        infectiousness=(
            NETWORK_MODEL_B_MULTIPLIER
        ),
    )
)


# ------------------------------------------------------------
# Same epidemic design on every network
# ------------------------------------------------------------

network_model_b_rng = (
    np.random.default_rng(
        NETWORK_MODEL_B_DESIGN_SEED
    )
)

network_model_b_seed_sets = [
    np.sort(
        network_model_b_rng.choice(
            N_NODES,
            size=NETWORK_MODEL_B_N_SEEDS,
            replace=False,
        )
    )
    for _ in range(
        NETWORK_MODEL_B_RUNS
    )
]

network_model_b_run_rng_seeds = (
    network_model_b_rng.integers(
        0,
        np.iinfo(np.int64).max,
        size=NETWORK_MODEL_B_RUNS,
        dtype=np.int64,
    )
)


# ------------------------------------------------------------
# Network-aware ensemble helper
#
# Do NOT use the older run_ensemble_from_design here,
# because that function refers to the global graph.
# ------------------------------------------------------------

def run_ensemble_on_network(
    current_graph,
    current_edge_multipliers,
    params,
    seed_sets,
    run_rng_seeds,
):

    return [
        simulate(
            current_graph,
            seeds=seeds,
            p=CALIBRATED_P,
            params=params,
            edge_multipliers=(
                current_edge_multipliers
            ),
            rng=int(run_seed),
        )
        for seeds, run_seed
        in zip(
            seed_sets,
            run_rng_seeds,
        )
    ]

In [27]:
# ============================================================
# Run Model B on each independent network realisation
# ============================================================

network_model_b_rows = []

network_model_b_results = {}


for network_seed in (
    NETWORK_REALISATION_SEEDS
):

    print(
        "\n"
        + "=" * 60
    )

    print(
        "NETWORK:",
        network_seed,
    )

    print(
        "=" * 60
    )

    current_graph = (
        network_realisations[
            network_seed
        ]["graph"]
    )

    current_edge_multipliers = (
        network_realisations[
            network_seed
        ]["edge_multipliers"]
    )

    # --------------------------------------------------------
    # Simulate matched control/planted ensembles
    # --------------------------------------------------------

    current_control_runs = (
        run_ensemble_on_network(
            current_graph,
            current_edge_multipliers,
            network_model_b_base_params,
            network_model_b_seed_sets,
            network_model_b_run_rng_seeds,
        )
    )

    current_planted_runs = (
        run_ensemble_on_network(
            current_graph,
            current_edge_multipliers,
            network_model_b_planted_params,
            network_model_b_seed_sets,
            network_model_b_run_rng_seeds,
        )
    )

    # --------------------------------------------------------
    # Record epidemic sizes BEFORE filtering
    # --------------------------------------------------------

    control_sizes_all = np.array(
        [
            run.final_size
            for run
            in current_control_runs
        ],
        dtype=np.int64,
    )

    planted_sizes_all = np.array(
        [
            run.final_size
            for run
            in current_planted_runs
        ],
        dtype=np.int64,
    )

    # --------------------------------------------------------
    # Apply the 1% rule while preserving matching
    # --------------------------------------------------------

    (
        current_control_runs,
        current_planted_runs,
        current_pair_keep,
    ) = filter_major_run_pairs(
        current_control_runs,
        current_planted_runs,
        control_label=(
            f"Network {network_seed} control"
        ),
        planted_label=(
            f"Network {network_seed} planted"
        ),
    )

    # --------------------------------------------------------
    # Model B observations
    # --------------------------------------------------------

    current_control_model_b = (
        model_b_observations(
            current_graph,
            current_control_runs,
            susceptibility=infection_rate,
            edge_multipliers=(
                current_edge_multipliers
            ),
        )
    )

    current_planted_model_b = (
        model_b_observations(
            current_graph,
            current_planted_runs,
            susceptibility=infection_rate,
            edge_multipliers=(
                current_edge_multipliers
            ),
        )
    )

    # --------------------------------------------------------
    # Full weighted Model B EMM search
    # --------------------------------------------------------

    current_control_results = (
        search_model_b_subgroups(
            people,
            current_control_model_b,
            categorical_columns=(
                NETWORK_MODEL_B_CATEGORICAL_COLUMNS
            ),
            age_column="age",
            age_thresholds=range(
                20,
                81,
                5,
            ),
            min_size=200,
            max_depth=2,
            direction="high",
            opportunity_type="weighted",
        )
    )

    current_planted_results = (
        search_model_b_subgroups(
            people,
            current_planted_model_b,
            categorical_columns=(
                NETWORK_MODEL_B_CATEGORICAL_COLUMNS
            ),
            age_column="age",
            age_thresholds=range(
                20,
                81,
                5,
            ),
            min_size=200,
            max_depth=2,
            direction="high",
            opportunity_type="weighted",
        )
    )

    # --------------------------------------------------------
    # Locate the known planted subgroup
    # --------------------------------------------------------

    planted_target = (
        current_planted_results[
            current_planted_results[
                "description"
            ]
            == NETWORK_MODEL_B_TARGET
        ]
    )

    control_target = (
        current_control_results[
            current_control_results[
                "description"
            ]
            == NETWORK_MODEL_B_TARGET
        ]
    )

    if len(planted_target) != 1:
        raise RuntimeError(
            "Could not uniquely locate "
            "the planted subgroup for "
            f"network {network_seed}."
        )

    if len(control_target) != 1:
        raise RuntimeError(
            "Could not uniquely locate "
            "the target subgroup in control "
            f"for network {network_seed}."
        )

    planted_target_index = (
        planted_target.index[0]
    )

    control_target_index = (
        control_target.index[0]
    )

    planted_rank = int(
        planted_target_index + 1
    )

    control_rank = int(
        control_target_index + 1
    )

    planted_rr = float(
        planted_target.iloc[0][
            "rate_ratio"
        ]
    )

    planted_quality = float(
        planted_target.iloc[0][
            "quality"
        ]
    )

    control_rr = float(
        control_target.iloc[0][
            "rate_ratio"
        ]
    )

    control_quality = float(
        control_target.iloc[0][
            "quality"
        ]
    )

    relative_error_pct = (
        abs(
            planted_rr
            - NETWORK_MODEL_B_MULTIPLIER
        )
        / NETWORK_MODEL_B_MULTIPLIER
        * 100.0
    )

    # --------------------------------------------------------
    # Save full results in case we need them later
    # --------------------------------------------------------

    network_model_b_results[
        network_seed
    ] = {
        "control_results":
            current_control_results,

        "planted_results":
            current_planted_results,

        "control_runs":
            current_control_runs,

        "planted_runs":
            current_planted_runs,
    }

    # --------------------------------------------------------
    # Summary row
    # --------------------------------------------------------

    network_model_b_rows.append(
        {
            "network_seed":
                network_seed,

            "control_attack_rate":
                float(
                    control_sizes_all.mean()
                    / N_NODES
                ),

            "planted_attack_rate":
                float(
                    planted_sizes_all.mean()
                    / N_NODES
                ),

            "pairs_retained":
                int(
                    current_pair_keep.sum()
                ),

            "planted_target_rank":
                planted_rank,

            "planted_target_rr":
                planted_rr,

            "planted_target_quality":
                planted_quality,

            "relative_error_pct":
                relative_error_pct,

            "control_target_rank":
                control_rank,

            "control_target_rr":
                control_rr,

            "control_target_quality":
                control_quality,
        }
    )

    print(
        "Planted rank:",
        planted_rank,
    )

    print(
        "Planted RR:",
        planted_rr,
    )

    print(
        "Relative error (%):",
        relative_error_pct,
    )

    print(
        "Control target RR:",
        control_rr,
    )


network_model_b_summary = (
    pd.DataFrame(
        network_model_b_rows
    )
)

display(
    network_model_b_summary
)


NETWORK: 22
Network 22 control minor outbreaks: 0/100 (0.00%)
Network 22 planted minor outbreaks: 0/100 (0.00%)
Matched pairs excluded: 0/100 (0.00%)
Matched pairs retained: 100
Planted rank: 1
Planted RR: 1.75858078761098
Relative error (%): 0.4903307206274299
Control target RR: 1.0036636853305978

NETWORK: 23
Network 23 control minor outbreaks: 0/100 (0.00%)
Network 23 planted minor outbreaks: 0/100 (0.00%)
Matched pairs excluded: 0/100 (0.00%)
Matched pairs retained: 100
Planted rank: 1
Planted RR: 1.747220067699764
Relative error (%): 0.1588532742991993
Control target RR: 1.0076982421255332

NETWORK: 24
Network 24 control minor outbreaks: 0/100 (0.00%)
Network 24 planted minor outbreaks: 0/100 (0.00%)
Matched pairs excluded: 0/100 (0.00%)
Matched pairs retained: 100
Planted rank: 1
Planted RR: 1.7634343615470207
Relative error (%): 0.7676778026868979
Control target RR: 1.0098099651963817

NETWORK: 25
Network 25 control minor outbreaks: 0/100 (0.00%)
Network 25 planted minor outbre

,network_seed,control_attack_rate,planted_attack_rate,pairs_retained,planted_target_rank,planted_target_rr,planted_target_quality,relative_error_pct,control_target_rank,control_target_rr,control_target_quality
0,22,0.392882,0.444011,100,1,1.758581,0.192347,0.490331,113,1.003664,0.001246
1,23,0.391175,0.442725,100,1,1.747220,0.190139,0.158853,34,1.007698,0.002613
2,24,0.394767,0.445097,100,1,1.763434,0.193286,0.767678,38,1.009810,0.003326
3,25,0.391569,0.443552,100,1,1.757146,0.192069,0.408357,247,0.997434,-0.000875
4,26,0.391425,0.444662,100,1,1.759178,0.192463,0.524471,30,1.009817,0.003329


In [28]:
# ============================================================
# Cross-network Model B summary
# ============================================================

print(
    "Networks:",
    len(
        network_model_b_summary
    ),
)

print(
    "\nExact target rank 1:",
    int(
        (
            network_model_b_summary[
                "planted_target_rank"
            ]
            == 1
        ).sum()
    ),
    "/",
    len(
        network_model_b_summary
    ),
)

print(
    "Top-5 recovery:",
    int(
        (
            network_model_b_summary[
                "planted_target_rank"
            ]
            <= 5
        ).sum()
    ),
    "/",
    len(
        network_model_b_summary
    ),
)

print(
    "\nMean planted RR:",
    network_model_b_summary[
        "planted_target_rr"
    ].mean(),
)

print(
    "SD planted RR:",
    network_model_b_summary[
        "planted_target_rr"
    ].std(
        ddof=1
    ),
)

print(
    "Min / max planted RR:",
    (
        network_model_b_summary[
            "planted_target_rr"
        ].min(),
        network_model_b_summary[
            "planted_target_rr"
        ].max(),
    ),
)

print(
    "\nMean relative error (%):",
    network_model_b_summary[
        "relative_error_pct"
    ].mean(),
)

print(
    "Mean control target RR:",
    network_model_b_summary[
        "control_target_rr"
    ].mean(),
)

print(
    "\nMean control attack rate:",
    network_model_b_summary[
        "control_attack_rate"
    ].mean(),
)

print(
    "Mean planted attack rate:",
    network_model_b_summary[
        "planted_attack_rate"
    ].mean(),
)

Networks: 5

Exact target rank 1: 5 / 5
Top-5 recovery: 5 / 5

Mean planted RR: 1.7571119413548957
SD planted RR: 0.006005215216182578
Min / max planted RR: (np.float64(1.747220067699764), np.float64(1.7634343615470207))

Mean relative error (%): 0.469937958570869
Mean control target RR: 1.0056847119341723

Mean control attack rate: 0.39236360000000003
Mean planted attack rate: 0.4440094


## 7. Model A cross-network validation


In [29]:
# ============================================================
# Model A validation across independent network realisations
# ============================================================

NETWORK_MODEL_A_RUNS = 100
NETWORK_MODEL_A_N_SEEDS = 50

NETWORK_MODEL_A_MULTIPLIER = 1.50

# Reproducible matched epidemic design used on every network.
NETWORK_MODEL_A_DESIGN_SEED = 20261023

NETWORK_MODEL_A_TARGET = (
    "smoking == 1 AND age >= 60"
)

NETWORK_MODEL_A_CATEGORICAL_COLUMNS = [
    "sex",
    "occupation",
    "smoking",
    "alcohol",
    "obesity",
]


# ------------------------------------------------------------
# Homogeneous baseline susceptibility
# ------------------------------------------------------------

NETWORK_MODEL_A_BASE_SUSCEPTIBILITY = float(
    infection_rate.mean()
)

print(
    "Model A baseline susceptibility:",
    NETWORK_MODEL_A_BASE_SUSCEPTIBILITY,
)


network_model_a_base_params = NodeParams(
    susceptibility=np.full(
        N_NODES,
        NETWORK_MODEL_A_BASE_SUSCEPTIBILITY,
        dtype=np.float64,
    ),
    infectiousness=np.ones(
        N_NODES,
        dtype=np.float64,
    ),
    recovery=np.full(
        N_NODES,
        RECOVERY_PROB,
        dtype=np.float64,
    ),
)


network_model_a_planted_params = (
    network_model_a_base_params.scaled(
        planted_mask,
        susceptibility=(
            NETWORK_MODEL_A_MULTIPLIER
        ),
    )
)


# ------------------------------------------------------------
# Same epidemic design on every network
# ------------------------------------------------------------

network_model_a_rng = (
    np.random.default_rng(
        NETWORK_MODEL_A_DESIGN_SEED
    )
)

network_model_a_seed_sets = [
    np.sort(
        network_model_a_rng.choice(
            N_NODES,
            size=NETWORK_MODEL_A_N_SEEDS,
            replace=False,
        )
    )
    for _ in range(
        NETWORK_MODEL_A_RUNS
    )
]

network_model_a_run_rng_seeds = (
    network_model_a_rng.integers(
        0,
        np.iinfo(np.int64).max,
        size=NETWORK_MODEL_A_RUNS,
        dtype=np.int64,
    )
)

Model A baseline susceptibility: 0.33760612


In [30]:
# ============================================================
# Run Model A on each independent network realisation
# ============================================================

network_model_a_rows = []

network_model_a_results = {}


for network_seed in (
    NETWORK_REALISATION_SEEDS
):

    print(
        "\n"
        + "=" * 60
    )

    print(
        "MODEL A NETWORK:",
        network_seed,
    )

    print(
        "=" * 60
    )

    current_graph = (
        network_realisations[
            network_seed
        ]["graph"]
    )

    current_edge_multipliers = (
        network_realisations[
            network_seed
        ]["edge_multipliers"]
    )

    # --------------------------------------------------------
    # Matched simulations
    # --------------------------------------------------------

    current_control_runs = (
        run_ensemble_on_network(
            current_graph,
            current_edge_multipliers,
            network_model_a_base_params,
            network_model_a_seed_sets,
            network_model_a_run_rng_seeds,
        )
    )

    current_planted_runs = (
        run_ensemble_on_network(
            current_graph,
            current_edge_multipliers,
            network_model_a_planted_params,
            network_model_a_seed_sets,
            network_model_a_run_rng_seeds,
        )
    )

    # --------------------------------------------------------
    # Epidemic size BEFORE filtering
    # --------------------------------------------------------

    control_sizes_all = np.array(
        [
            run.final_size
            for run
            in current_control_runs
        ],
        dtype=np.int64,
    )

    planted_sizes_all = np.array(
        [
            run.final_size
            for run
            in current_planted_runs
        ],
        dtype=np.int64,
    )

    # --------------------------------------------------------
    # Apply 1% minor-outbreak rule
    # while preserving matched pairs
    # --------------------------------------------------------

    (
        current_control_runs,
        current_planted_runs,
        current_pair_keep,
    ) = filter_major_run_pairs(
        current_control_runs,
        current_planted_runs,
        control_label=(
            f"Model A network "
            f"{network_seed} control"
        ),
        planted_label=(
            f"Model A network "
            f"{network_seed} planted"
        ),
    )

    # --------------------------------------------------------
    # Build Model A observations
    # --------------------------------------------------------

    current_control_obs = (
        model_a_observations(
            current_graph,
            current_control_runs,
            infectiousness=(
                network_model_a_base_params
                .infectiousness
            ),
            edge_multipliers=(
                current_edge_multipliers
            ),
        )
    )

    current_planted_obs = (
        model_a_observations(
            current_graph,
            current_planted_runs,
            infectiousness=(
                network_model_a_planted_params
                .infectiousness
            ),
            edge_multipliers=(
                current_edge_multipliers
            ),
        )
    )

    # --------------------------------------------------------
    # Full Model A EMM search
    # --------------------------------------------------------

    current_control_results = (
        search_model_a_subgroups(
            people,
            current_control_obs,
            categorical_columns=(
                NETWORK_MODEL_A_CATEGORICAL_COLUMNS
            ),
            age_column="age",
            age_thresholds=range(
                20,
                81,
                5,
            ),
            min_size=200,
            max_depth=2,
            direction="high",
            exposure_type="weighted",
        )
    )

    current_planted_results = (
        search_model_a_subgroups(
            people,
            current_planted_obs,
            categorical_columns=(
                NETWORK_MODEL_A_CATEGORICAL_COLUMNS
            ),
            age_column="age",
            age_thresholds=range(
                20,
                81,
                5,
            ),
            min_size=200,
            max_depth=2,
            direction="high",
            exposure_type="weighted",
        )
    )

    # --------------------------------------------------------
    # Locate the planted subgroup
    # --------------------------------------------------------

    planted_target = (
        current_planted_results[
            current_planted_results[
                "description"
            ]
            == NETWORK_MODEL_A_TARGET
        ]
    )

    control_target = (
        current_control_results[
            current_control_results[
                "description"
            ]
            == NETWORK_MODEL_A_TARGET
        ]
    )

    if len(planted_target) != 1:
        raise RuntimeError(
            "Could not uniquely locate "
            "the planted Model A subgroup "
            f"for network {network_seed}."
        )

    if len(control_target) != 1:
        raise RuntimeError(
            "Could not uniquely locate "
            "the target Model A subgroup "
            f"in control for network "
            f"{network_seed}."
        )

    planted_rank = int(
        planted_target.index[0] + 1
    )

    control_rank = int(
        control_target.index[0] + 1
    )

    planted_rr = float(
        planted_target.iloc[0][
            "rate_ratio"
        ]
    )

    planted_quality = float(
        planted_target.iloc[0][
            "quality"
        ]
    )

    control_rr = float(
        control_target.iloc[0][
            "rate_ratio"
        ]
    )

    control_quality = float(
        control_target.iloc[0][
            "quality"
        ]
    )

    relative_error_pct = (
        abs(
            planted_rr
            - NETWORK_MODEL_A_MULTIPLIER
        )
        / NETWORK_MODEL_A_MULTIPLIER
        * 100.0
    )

    # --------------------------------------------------------
    # Save search results
    # --------------------------------------------------------

    network_model_a_results[
        network_seed
    ] = {
        "control_results":
            current_control_results,

        "planted_results":
            current_planted_results,

        # Retain the filtered matched runs so later
        # ablation analyses can reuse the exact same
        # simulations without rerunning the epidemic.
        "control_runs":
            current_control_runs,

        "planted_runs":
            current_planted_runs,
    }

    # --------------------------------------------------------
    # Summary row
    # --------------------------------------------------------

    network_model_a_rows.append(
        {
            "network_seed":
                network_seed,

            "control_attack_rate":
                float(
                    control_sizes_all.mean()
                    / N_NODES
                ),

            "planted_attack_rate":
                float(
                    planted_sizes_all.mean()
                    / N_NODES
                ),

            "pairs_retained":
                int(
                    current_pair_keep.sum()
                ),

            "planted_target_rank":
                planted_rank,

            "planted_target_rr":
                planted_rr,

            "planted_target_quality":
                planted_quality,

            "relative_error_pct":
                relative_error_pct,

            "control_target_rank":
                control_rank,

            "control_target_rr":
                control_rr,

            "control_target_quality":
                control_quality,
        }
    )

    print(
        "Planted rank:",
        planted_rank,
    )

    print(
        "Planted RR:",
        planted_rr,
    )

    print(
        "Relative error (%):",
        relative_error_pct,
    )

    print(
        "Control target rank:",
        control_rank,
    )

    print(
        "Control target RR:",
        control_rr,
    )


network_model_a_summary = (
    pd.DataFrame(
        network_model_a_rows
    )
)

display(
    network_model_a_summary
)


MODEL A NETWORK: 22
Model A network 22 control minor outbreaks: 0/100 (0.00%)
Model A network 22 planted minor outbreaks: 0/100 (0.00%)
Matched pairs excluded: 0/100 (0.00%)
Matched pairs retained: 100
Planted rank: 1
Planted RR: 1.5147204019645926
Relative error (%): 0.9813601309728378
Control target rank: 279
Control target RR: 0.9948277767316399

MODEL A NETWORK: 23
Model A network 23 control minor outbreaks: 0/100 (0.00%)
Model A network 23 planted minor outbreaks: 0/100 (0.00%)
Matched pairs excluded: 0/100 (0.00%)
Matched pairs retained: 100
Planted rank: 1
Planted RR: 1.5206086451521932
Relative error (%): 1.373909676812879
Control target rank: 233
Control target RR: 0.9988103783428817

MODEL A NETWORK: 24
Model A network 24 control minor outbreaks: 0/100 (0.00%)
Model A network 24 planted minor outbreaks: 0/100 (0.00%)
Matched pairs excluded: 0/100 (0.00%)
Matched pairs retained: 100
Planted rank: 1
Planted RR: 1.515929559097998
Relative error (%): 1.0619706065331989
Control t

,network_seed,control_attack_rate,planted_attack_rate,pairs_retained,planted_target_rank,planted_target_rr,planted_target_quality,relative_error_pct,control_target_rank,control_target_rr,control_target_quality
0,22,0.448876,0.477638,100,1,1.514720,0.141483,0.981360,279,0.994828,-0.001767
1,23,0.450097,0.477547,100,1,1.520609,0.142805,1.373910,233,0.998810,-0.000406
2,24,0.449863,0.479777,100,1,1.515930,0.141755,1.061971,362,0.990688,-0.003188
3,25,0.449574,0.479883,100,1,1.514312,0.141391,0.954101,269,0.998046,-0.000666
4,26,0.451812,0.479232,100,1,1.509146,0.140227,0.609712,14,1.008463,0.002871


In [31]:
# ============================================================
# Cross-network Model A summary
# ============================================================

print(
    "Networks:",
    len(
        network_model_a_summary
    ),
)

print(
    "\nExact target rank 1:",
    int(
        (
            network_model_a_summary[
                "planted_target_rank"
            ]
            == 1
        ).sum()
    ),
    "/",
    len(
        network_model_a_summary
    ),
)

print(
    "Top-5 recovery:",
    int(
        (
            network_model_a_summary[
                "planted_target_rank"
            ]
            <= 5
        ).sum()
    ),
    "/",
    len(
        network_model_a_summary
    ),
)

print(
    "\nMean planted RR:",
    network_model_a_summary[
        "planted_target_rr"
    ].mean(),
)

print(
    "SD planted RR:",
    network_model_a_summary[
        "planted_target_rr"
    ].std(
        ddof=1
    ),
)

print(
    "Min / max planted RR:",
    (
        network_model_a_summary[
            "planted_target_rr"
        ].min(),
        network_model_a_summary[
            "planted_target_rr"
        ].max(),
    ),
)

print(
    "\nMean relative error (%):",
    network_model_a_summary[
        "relative_error_pct"
    ].mean(),
)

print(
    "Mean control target RR:",
    network_model_a_summary[
        "control_target_rr"
    ].mean(),
)

print(
    "\nMean control attack rate:",
    network_model_a_summary[
        "control_attack_rate"
    ].mean(),
)

print(
    "Mean planted attack rate:",
    network_model_a_summary[
        "planted_attack_rate"
    ].mean(),
)

Networks: 5

Exact target rank 1: 5 / 5
Top-5 recovery: 5 / 5

Mean planted RR: 1.5149431598030065
SD planted RR: 0.004096646362670143
Min / max planted RR: (np.float64(1.5091456832119556), np.float64(1.5206086451521932))

Mean relative error (%): 0.996210653533763
Mean control target RR: 0.9981669102066217

Mean control attack rate: 0.4500444
Mean planted attack rate: 0.47881540000000006


## 8. Mechanism stress tests


In [32]:
# ============================================================
# Stress-test plants:
# 1. Dead-end
# 2. Adversarial spreader
# ============================================================

STRESS_TEST_RUNS = 100
STRESS_TEST_N_SEEDS = 50
STRESS_TEST_SEED = 20261026

STRESS_TARGET = (
    "smoking == 1 AND age >= 60"
)

STRESS_CATEGORICAL_COLUMNS = [
    "sex",
    "occupation",
    "smoking",
    "alcohol",
    "obesity",
]


# ------------------------------------------------------------
# Common clean baseline
#
# Homogeneous susceptibility gives known ground truth:
# subgroup susceptibility effects are exactly the planted
# multipliers.
# ------------------------------------------------------------

STRESS_BASE_SUSCEPTIBILITY = float(
    infection_rate.mean()
)

stress_base_params = NodeParams(
    susceptibility=np.full(
        N_NODES,
        STRESS_BASE_SUSCEPTIBILITY,
        dtype=np.float64,
    ),
    infectiousness=np.ones(
        N_NODES,
        dtype=np.float64,
    ),
    recovery=np.full(
        N_NODES,
        RECOVERY_PROB,
        dtype=np.float64,
    ),
)


# ============================================================
# DEAD-END PLANT
#
# Easier to infect, but poor transmitter.
# ============================================================

DEAD_END_SUSCEPTIBILITY = 1.50
DEAD_END_INFECTIOUSNESS = 0.50

dead_end_params = (
    stress_base_params.scaled(
        planted_mask,
        susceptibility=(
            DEAD_END_SUSCEPTIBILITY
        ),
        infectiousness=(
            DEAD_END_INFECTIOUSNESS
        ),
    )
)


# ============================================================
# ADVERSARIAL SPREADER
#
# Harder to infect, but much stronger transmitter.
# ============================================================

ADVERSARIAL_SUSCEPTIBILITY = 0.60
ADVERSARIAL_INFECTIOUSNESS = 1.75

adversarial_params = (
    stress_base_params.scaled(
        planted_mask,
        susceptibility=(
            ADVERSARIAL_SUSCEPTIBILITY
        ),
        infectiousness=(
            ADVERSARIAL_INFECTIOUSNESS
        ),
    )
)


print(
    "Dead-end true susceptibility RR:",
    DEAD_END_SUSCEPTIBILITY,
)

print(
    "Dead-end true infectiousness RR:",
    DEAD_END_INFECTIOUSNESS,
)

print(
    "Adversarial true susceptibility RR:",
    ADVERSARIAL_SUSCEPTIBILITY,
)

print(
    "Adversarial true infectiousness RR:",
    ADVERSARIAL_INFECTIOUSNESS,
)

Dead-end true susceptibility RR: 1.5
Dead-end true infectiousness RR: 0.5
Adversarial true susceptibility RR: 0.6
Adversarial true infectiousness RR: 1.75


In [33]:
# ============================================================
# Matched stress-test design
# ============================================================

stress_rng = np.random.default_rng(
    STRESS_TEST_SEED
)

stress_seed_sets = [
    np.sort(
        stress_rng.choice(
            N_NODES,
            size=STRESS_TEST_N_SEEDS,
            replace=False,
        )
    )
    for _ in range(
        STRESS_TEST_RUNS
    )
]

stress_run_rng_seeds = (
    stress_rng.integers(
        0,
        np.iinfo(np.int64).max,
        size=STRESS_TEST_RUNS,
        dtype=np.int64,
    )
)


stress_control_runs = (
    run_ensemble_on_network(
        graph,
        edge_multipliers,
        stress_base_params,
        stress_seed_sets,
        stress_run_rng_seeds,
    )
)

dead_end_runs = (
    run_ensemble_on_network(
        graph,
        edge_multipliers,
        dead_end_params,
        stress_seed_sets,
        stress_run_rng_seeds,
    )
)

adversarial_runs = (
    run_ensemble_on_network(
        graph,
        edge_multipliers,
        adversarial_params,
        stress_seed_sets,
        stress_run_rng_seeds,
    )
)


print(
    "Control mean final size:",
    np.mean(
        [
            r.final_size
            for r in stress_control_runs
        ]
    ),
)

print(
    "Dead-end mean final size:",
    np.mean(
        [
            r.final_size
            for r in dead_end_runs
        ]
    ),
)

print(
    "Adversarial mean final size:",
    np.mean(
        [
            r.final_size
            for r in adversarial_runs
        ]
    ),
)

Control mean final size: 4506.56
Dead-end mean final size: 4461.03
Adversarial mean final size: 4443.55


In [34]:
# ============================================================
# Minor-outbreak filtering
# ============================================================

(
    dead_end_control_runs,
    dead_end_runs_filtered,
    dead_end_pair_mask,
) = filter_major_run_pairs(
    stress_control_runs,
    dead_end_runs,
    control_label="Dead-end control",
    planted_label="Dead-end planted",
)


(
    adversarial_control_runs,
    adversarial_runs_filtered,
    adversarial_pair_mask,
) = filter_major_run_pairs(
    stress_control_runs,
    adversarial_runs,
    control_label="Adversarial control",
    planted_label="Adversarial planted",
)

Dead-end control minor outbreaks: 0/100 (0.00%)
Dead-end planted minor outbreaks: 0/100 (0.00%)
Matched pairs excluded: 0/100 (0.00%)
Matched pairs retained: 100
Adversarial control minor outbreaks: 0/100 (0.00%)
Adversarial planted minor outbreaks: 0/100 (0.00%)
Matched pairs excluded: 0/100 (0.00%)
Matched pairs retained: 100


In [35]:
# ============================================================
# DEAD-END: Model A
# Expected truth: susceptibility RR = 1.50
# ============================================================

dead_end_model_a_obs = (
    model_a_observations(
        graph,
        dead_end_runs_filtered,
        infectiousness=(
            dead_end_params.infectiousness
        ),
        edge_multipliers=edge_multipliers,
    )
)

dead_end_model_a_results = (
    search_model_a_subgroups(
        people,
        dead_end_model_a_obs,
        categorical_columns=(
            STRESS_CATEGORICAL_COLUMNS
        ),
        age_column="age",
        age_thresholds=range(
            20,
            81,
            5,
        ),
        min_size=200,
        max_depth=2,
        direction="high",
        exposure_type="weighted",
    )
)

dead_end_a_target = (
    dead_end_model_a_results[
        dead_end_model_a_results[
            "description"
        ]
        == STRESS_TARGET
    ]
)

print(
    "Dead-end Model A rank:",
    dead_end_a_target.index[0] + 1,
)

print(
    "Estimated susceptibility RR:",
    dead_end_a_target.iloc[0][
        "rate_ratio"
    ],
)

print(
    "True susceptibility RR:",
    DEAD_END_SUSCEPTIBILITY,
)

Dead-end Model A rank: 1
Estimated susceptibility RR: 1.5122663812450512
True susceptibility RR: 1.5


In [36]:
# ============================================================
# DEAD-END: Model B
# Expected truth: infectiousness RR = 0.50
# ============================================================

dead_end_model_b_obs = (
    model_b_observations(
        graph,
        dead_end_runs_filtered,
        susceptibility=(
            dead_end_params.susceptibility
        ),
        edge_multipliers=edge_multipliers,
    )
)

dead_end_model_b_results = (
    search_model_b_subgroups(
        people,
        dead_end_model_b_obs,
        categorical_columns=(
            STRESS_CATEGORICAL_COLUMNS
        ),
        age_column="age",
        age_thresholds=range(
            20,
            81,
            5,
        ),
        min_size=200,
        max_depth=2,
        direction="low",
        opportunity_type="weighted",
    )
)

dead_end_b_target = (
    dead_end_model_b_results[
        dead_end_model_b_results[
            "description"
        ]
        == STRESS_TARGET
    ]
)

print(
    "Dead-end Model B rank:",
    dead_end_b_target.index[0] + 1,
)

print(
    "Estimated infectiousness RR:",
    dead_end_b_target.iloc[0][
        "rate_ratio"
    ],
)

print(
    "True infectiousness RR:",
    DEAD_END_INFECTIOUSNESS,
)

Dead-end Model B rank: 1
Estimated infectiousness RR: 0.4991235266749672
True infectiousness RR: 0.5


In [37]:
# ============================================================
# ADVERSARIAL SPREADER: Model A
# Expected truth: susceptibility RR = 0.60
# ============================================================

adversarial_model_a_obs = (
    model_a_observations(
        graph,
        adversarial_runs_filtered,
        infectiousness=(
            adversarial_params.infectiousness
        ),
        edge_multipliers=edge_multipliers,
    )
)

adversarial_model_a_results = (
    search_model_a_subgroups(
        people,
        adversarial_model_a_obs,
        categorical_columns=(
            STRESS_CATEGORICAL_COLUMNS
        ),
        age_column="age",
        age_thresholds=range(
            20,
            81,
            5,
        ),
        min_size=200,
        max_depth=2,
        direction="low",
        exposure_type="weighted",
    )
)

adversarial_a_target = (
    adversarial_model_a_results[
        adversarial_model_a_results[
            "description"
        ]
        == STRESS_TARGET
    ]
)

print(
    "Adversarial Model A rank:",
    adversarial_a_target.index[0] + 1,
)

print(
    "Estimated susceptibility RR:",
    adversarial_a_target.iloc[0][
        "rate_ratio"
    ],
)

print(
    "True susceptibility RR:",
    ADVERSARIAL_SUSCEPTIBILITY,
)

Adversarial Model A rank: 1
Estimated susceptibility RR: 0.593886609477968
True susceptibility RR: 0.6


In [38]:
# ============================================================
# ADVERSARIAL SPREADER: Model B
# Expected truth: infectiousness RR = 1.75
# ============================================================

adversarial_model_b_obs = (
    model_b_observations(
        graph,
        adversarial_runs_filtered,
        susceptibility=(
            adversarial_params.susceptibility
        ),
        edge_multipliers=edge_multipliers,
    )
)

adversarial_model_b_results = (
    search_model_b_subgroups(
        people,
        adversarial_model_b_obs,
        categorical_columns=(
            STRESS_CATEGORICAL_COLUMNS
        ),
        age_column="age",
        age_thresholds=range(
            20,
            81,
            5,
        ),
        min_size=200,
        max_depth=2,
        direction="high",
        opportunity_type="weighted",
    )
)

adversarial_b_target = (
    adversarial_model_b_results[
        adversarial_model_b_results[
            "description"
        ]
        == STRESS_TARGET
    ]
)

print(
    "Adversarial Model B rank:",
    adversarial_b_target.index[0] + 1,
)

print(
    "Estimated infectiousness RR:",
    adversarial_b_target.iloc[0][
        "rate_ratio"
    ],
)

print(
    "True infectiousness RR:",
    ADVERSARIAL_INFECTIOUSNESS,
)

Adversarial Model B rank: 1
Estimated infectiousness RR: 1.7594388604487645
True infectiousness RR: 1.75


In [39]:
# ============================================================
# Helper: infection frequency inside vs outside subgroup
# ============================================================

def infection_frequency_by_group(
    runs,
    mask,
):

    infected = np.stack(
        [
            run.infection_time >= 0
            for run in runs
        ]
    )

    person_frequency = (
        infected.mean(axis=0)
    )

    return {
        "group":
            float(
                person_frequency[
                    mask
                ].mean()
            ),

        "rest":
            float(
                person_frequency[
                    ~mask
                ].mean()
            ),
    }

In [40]:
dead_end_naive = (
    infection_frequency_by_group(
        dead_end_runs_filtered,
        planted_mask,
    )
)

adversarial_naive = (
    infection_frequency_by_group(
        adversarial_runs_filtered,
        planted_mask,
    )
)

print(
    "Dead-end infection frequency:",
    dead_end_naive,
)

print(
    "Adversarial infection frequency:",
    adversarial_naive,
)

Dead-end infection frequency: {'group': 0.3211886304909561, 'rest': 0.4625104649847268}
Adversarial infection frequency: {'group': 0.17291989664082688, 'rest': 0.48000791944790133}


In [41]:
stress_test_summary = pd.DataFrame(
    [
        {
            "scenario": "Dead-end",
            "true_susceptibility_rr":
                DEAD_END_SUSCEPTIBILITY,
            "estimated_model_a_rr":
                float(
                    dead_end_a_target.iloc[0][
                        "rate_ratio"
                    ]
                ),
            "model_a_rank":
                int(
                    dead_end_a_target.index[0] + 1
                ),
            "true_infectiousness_rr":
                DEAD_END_INFECTIOUSNESS,
            "estimated_model_b_rr":
                float(
                    dead_end_b_target.iloc[0][
                        "rate_ratio"
                    ]
                ),
            "model_b_rank":
                int(
                    dead_end_b_target.index[0] + 1
                ),
            "naive_group_infection_frequency":
                dead_end_naive["group"],
            "naive_rest_infection_frequency":
                dead_end_naive["rest"],
        },
        {
            "scenario": "Adversarial spreader",
            "true_susceptibility_rr":
                ADVERSARIAL_SUSCEPTIBILITY,
            "estimated_model_a_rr":
                float(
                    adversarial_a_target.iloc[0][
                        "rate_ratio"
                    ]
                ),
            "model_a_rank":
                int(
                    adversarial_a_target.index[0] + 1
                ),
            "true_infectiousness_rr":
                ADVERSARIAL_INFECTIOUSNESS,
            "estimated_model_b_rr":
                float(
                    adversarial_b_target.iloc[0][
                        "rate_ratio"
                    ]
                ),
            "model_b_rank":
                int(
                    adversarial_b_target.index[0] + 1
                ),
            "naive_group_infection_frequency":
                adversarial_naive["group"],
            "naive_rest_infection_frequency":
                adversarial_naive["rest"],
        },
    ]
)

display(stress_test_summary)

,scenario,true_susceptibility_rr,estimated_model_a_rr,model_a_rank,true_infectiousness_rr,estimated_model_b_rr,model_b_rank,naive_group_infection_frequency,naive_rest_infection_frequency
0,Dead-end,1.5,1.512266,1,0.50,0.499124,1,0.321189,0.462510
1,Adversarial spreader,0.6,0.593887,1,1.75,1.759439,1,0.172920,0.480008


## 9. Model B opportunity-weighting ablation

This ablation isolates the contribution of the opportunity weighting used by Model B on the primary network (`network_seed = 22`).

The same final matched epidemic runs used in the cross-network validation are reused here; no new epidemic simulations are generated.

Three opportunity definitions are compared for the known planted subgroup:

1. **Raw opportunity** — susceptible contact opportunities are counted without susceptibility or household-edge weighting.
2. **Susceptibility weighted** — opportunities are weighted by the susceptible neighbour's baseline susceptibility, but household-edge multipliers are omitted.
3. **Susceptibility + edge weighted** — the final Model B definition, incorporating both susceptibility and edge-specific transmission intensity.

For the planted condition, the target infectiousness rate ratio is 1.75. For the control condition, the null target is 1.00.


In [42]:
# ============================================================
# Model B opportunity-weighting ablation
# Reuse final network-22 matched runs
# ============================================================

ABLATION_NETWORK_SEED = 22

ablation_graph = (
    network_realisations[
        ABLATION_NETWORK_SEED
    ]["graph"]
)

ablation_edge_multipliers = (
    network_realisations[
        ABLATION_NETWORK_SEED
    ]["edge_multipliers"]
)

model_b_ablation_control_runs = (
    network_model_b_results[
        ABLATION_NETWORK_SEED
    ]["control_runs"]
)

model_b_ablation_planted_runs = (
    network_model_b_results[
        ABLATION_NETWORK_SEED
    ]["planted_runs"]
)


# ------------------------------------------------------------
# Full final Model B observations:
# susceptibility + edge weighting available
# ------------------------------------------------------------

model_b_control_full_obs = (
    model_b_observations(
        ablation_graph,
        model_b_ablation_control_runs,
        susceptibility=infection_rate,
        edge_multipliers=(
            ablation_edge_multipliers
        ),
    )
)

model_b_planted_full_obs = (
    model_b_observations(
        ablation_graph,
        model_b_ablation_planted_runs,
        susceptibility=infection_rate,
        edge_multipliers=(
            ablation_edge_multipliers
        ),
    )
)


# ------------------------------------------------------------
# Susceptibility-only observations:
# all contact edges effectively receive weight 1
# ------------------------------------------------------------

model_b_control_sus_only_obs = (
    model_b_observations(
        ablation_graph,
        model_b_ablation_control_runs,
        susceptibility=infection_rate,
    )
)

model_b_planted_sus_only_obs = (
    model_b_observations(
        ablation_graph,
        model_b_ablation_planted_runs,
        susceptibility=infection_rate,
    )
)

print(
    "Model B ablation network:",
    ABLATION_NETWORK_SEED,
)

print(
    "Matched control runs reused:",
    len(model_b_ablation_control_runs),
)

print(
    "Matched planted runs reused:",
    len(model_b_ablation_planted_runs),
)

Model B ablation network: 22
Matched control runs reused: 100
Matched planted runs reused: 100


In [43]:
# ============================================================
# Score the exact target under each Model B opportunity
# definition
# ============================================================

MODEL_B_ABLATION_TRUE_PLANTED_RR = (
    NETWORK_MODEL_B_MULTIPLIER
)


def model_b_ablation_scores(
    full_obs,
    sus_only_obs,
):
    return {
        "Raw opportunity":
            score_model_b_subgroup(
                planted_mask,
                full_obs,
                min_size=200,
                direction="high",
                opportunity_type="raw",
            ),

        "Susceptibility weighted":
            score_model_b_subgroup(
                planted_mask,
                sus_only_obs,
                min_size=200,
                direction="high",
                opportunity_type="weighted",
            ),

        "Susceptibility + edge weighted":
            score_model_b_subgroup(
                planted_mask,
                full_obs,
                min_size=200,
                direction="high",
                opportunity_type="weighted",
            ),
    }


model_b_planted_ablation_scores = (
    model_b_ablation_scores(
        model_b_planted_full_obs,
        model_b_planted_sus_only_obs,
    )
)

model_b_control_ablation_scores = (
    model_b_ablation_scores(
        model_b_control_full_obs,
        model_b_control_sus_only_obs,
    )
)


model_b_ablation_rows = []

for condition, scores, target_rr in [
    (
        "Planted",
        model_b_planted_ablation_scores,
        MODEL_B_ABLATION_TRUE_PLANTED_RR,
    ),
    (
        "Control",
        model_b_control_ablation_scores,
        1.0,
    ),
]:

    for method, score in scores.items():

        estimated_rr = float(
            score["rate_ratio"]
        )

        absolute_error = abs(
            estimated_rr
            - target_rr
        )

        model_b_ablation_rows.append(
            {
                "condition":
                    condition,

                "method":
                    method,

                "rate_ratio":
                    estimated_rr,

                "quality":
                    float(
                        score["quality"]
                    ),

                "target_rr":
                    target_rr,

                "absolute_RR_error":
                    absolute_error,

                "relative_error_percent":
                    (
                        absolute_error
                        / target_rr
                        * 100.0
                    ),
            }
        )


model_b_weighting_ablation = (
    pd.DataFrame(
        model_b_ablation_rows
    )
)

display(
    model_b_weighting_ablation
)

,condition,method,rate_ratio,quality,target_rr,absolute_RR_error,relative_error_percent
0,Planted,Raw opportunity,1.906413,0.219850,1.75,0.156413,8.937904
1,Planted,Susceptibility weighted,1.691309,0.179057,1.75,0.058691,3.353764
2,Planted,Susceptibility + edge weighted,1.758581,0.192347,1.75,0.008581,0.490331
3,Control,Raw opportunity,1.092377,0.030106,1.00,0.092377,9.237717
4,Control,Susceptibility weighted,0.966885,-0.011475,1.00,0.033115,3.311525
5,Control,Susceptibility + edge weighted,1.003664,0.001246,1.00,0.003664,0.366369


In [44]:
# Compact comparison for the planted condition.
model_b_planted_comparison = (
    model_b_weighting_ablation[
        model_b_weighting_ablation[
            "condition"
        ]
        == "Planted"
    ][
        [
            "method",
            "rate_ratio",
            "target_rr",
            "absolute_RR_error",
            "relative_error_percent",
        ]
    ]
    .reset_index(drop=True)
)

display(
    model_b_planted_comparison
)

best_model_b_method = (
    model_b_planted_comparison
    .sort_values(
        "absolute_RR_error"
    )
    .iloc[0]["method"]
)

print(
    "Closest Model B estimate to the planted truth:",
    best_model_b_method,
)

,method,rate_ratio,target_rr,absolute_RR_error,relative_error_percent
0,Raw opportunity,1.906413,1.75,0.156413,8.937904
1,Susceptibility weighted,1.691309,1.75,0.058691,3.353764
2,Susceptibility + edge weighted,1.758581,1.75,0.008581,0.490331


Closest Model B estimate to the planted truth: Susceptibility + edge weighted


## 10. Model A exposure-weighting ablation

This ablation tests whether Model A benefits from using edge-weighted infectious exposure rather than raw infectious-contact exposure.

Again, the primary network (`network_seed = 22`) and the exact matched Model A runs from the final validation are reused, so the comparison changes only the exposure definition.

Two scoring variants are compared:

1. **Raw exposure** — infectious neighbours contribute equally.
2. **Weighted exposure** — infectious exposure is multiplied by the edge-specific transmission intensity, so household contacts receive the final household multiplier.

The planted susceptibility rate ratio is 1.50, while the corresponding control target is 1.00.


In [45]:
# ============================================================
# Model A exposure-weighting ablation
# Reuse final network-22 matched runs
# ============================================================

model_a_ablation_control_runs = (
    network_model_a_results[
        ABLATION_NETWORK_SEED
    ]["control_runs"]
)

model_a_ablation_planted_runs = (
    network_model_a_results[
        ABLATION_NETWORK_SEED
    ]["planted_runs"]
)


# The observation object contains both raw and weighted
# exposure quantities; the scoring step below chooses
# which exposure definition to use.
model_a_control_ablation_obs = (
    model_a_observations(
        ablation_graph,
        model_a_ablation_control_runs,
        infectiousness=(
            network_model_a_base_params
            .infectiousness
        ),
        edge_multipliers=(
            ablation_edge_multipliers
        ),
    )
)

model_a_planted_ablation_obs = (
    model_a_observations(
        ablation_graph,
        model_a_ablation_planted_runs,
        infectiousness=(
            network_model_a_planted_params
            .infectiousness
        ),
        edge_multipliers=(
            ablation_edge_multipliers
        ),
    )
)

print(
    "Model A ablation network:",
    ABLATION_NETWORK_SEED,
)

print(
    "Matched control runs reused:",
    len(model_a_ablation_control_runs),
)

print(
    "Matched planted runs reused:",
    len(model_a_ablation_planted_runs),
)

Model A ablation network: 22
Matched control runs reused: 100
Matched planted runs reused: 100


In [46]:
# ============================================================
# Score the exact target under raw and weighted Model A
# exposure definitions
# ============================================================

MODEL_A_ABLATION_TRUE_PLANTED_RR = (
    NETWORK_MODEL_A_MULTIPLIER
)


def model_a_ablation_scores(
    observations,
):
    return {
        "Raw":
            score_model_a_subgroup(
                planted_mask,
                observations,
                min_size=200,
                direction="high",
                exposure_type="raw",
            ),

        "Weighted":
            score_model_a_subgroup(
                planted_mask,
                observations,
                min_size=200,
                direction="high",
                exposure_type="weighted",
            ),
    }


model_a_planted_ablation_scores = (
    model_a_ablation_scores(
        model_a_planted_ablation_obs
    )
)

model_a_control_ablation_scores = (
    model_a_ablation_scores(
        model_a_control_ablation_obs
    )
)


model_a_ablation_rows = []

for condition, scores, target_rr in [
    (
        "Planted",
        model_a_planted_ablation_scores,
        MODEL_A_ABLATION_TRUE_PLANTED_RR,
    ),
    (
        "Control",
        model_a_control_ablation_scores,
        1.0,
    ),
]:

    for exposure, score in scores.items():

        estimated_rr = float(
            score["rate_ratio"]
        )

        absolute_error = abs(
            estimated_rr
            - target_rr
        )

        model_a_ablation_rows.append(
            {
                "condition":
                    condition,

                "exposure":
                    exposure,

                "rate_ratio":
                    estimated_rr,

                "quality":
                    float(
                        score["quality"]
                    ),

                "target_rr":
                    target_rr,

                "absolute_RR_error":
                    absolute_error,

                "relative_error_percent":
                    (
                        absolute_error
                        / target_rr
                        * 100.0
                    ),
            }
        )


model_a_exposure_ablation = (
    pd.DataFrame(
        model_a_ablation_rows
    )
)

display(
    model_a_exposure_ablation
)

,condition,exposure,rate_ratio,quality,target_rr,absolute_RR_error,relative_error_percent
0,Planted,Raw,1.431971,0.122341,1.5,0.068029,4.535274
1,Planted,Weighted,1.514720,0.141483,1.5,0.014720,0.981360
2,Control,Raw,0.935850,-0.022591,1.0,0.064150,6.414957
3,Control,Weighted,0.994828,-0.001767,1.0,0.005172,0.517222


In [47]:
# Compact comparison for the planted condition.
model_a_planted_comparison = (
    model_a_exposure_ablation[
        model_a_exposure_ablation[
            "condition"
        ]
        == "Planted"
    ][
        [
            "exposure",
            "rate_ratio",
            "target_rr",
            "absolute_RR_error",
            "relative_error_percent",
        ]
    ]
    .reset_index(drop=True)
)

display(
    model_a_planted_comparison
)

best_model_a_exposure = (
    model_a_planted_comparison
    .sort_values(
        "absolute_RR_error"
    )
    .iloc[0]["exposure"]
)

print(
    "Closest Model A estimate to the planted truth:",
    best_model_a_exposure,
)

,exposure,rate_ratio,target_rr,absolute_RR_error,relative_error_percent
0,Raw,1.431971,1.5,0.068029,4.535274
1,Weighted,1.514720,1.5,0.014720,0.981360


Closest Model A estimate to the planted truth: Weighted


## 11. Naive outcome baselines

The contact-adjusted models are compared with simple outcome-based
subgroup discovery baselines that do not account for realised contact.

For susceptibility, the naive target is each person's infection
frequency across epidemic runs. For infectiousness, the naive target is
the mean number of secondary infections caused in runs in which the
person was infected.

The same descriptor language, age thresholds, minimum subgroup size and
maximum description depth are used as for Models A and B. This isolates
the effect of contact adjustment rather than differences in the search
space.

The stress-test scenarios are particularly informative: they test
whether raw outcomes can identify the planted mechanism when
susceptibility and infectiousness act in opposite directions.

In [53]:
# ============================================================
# Section 11: Naive outcome baselines
# ============================================================

NAIVE_CATEGORICAL_COLUMNS = [
    "sex",
    "occupation",
    "smoking",
    "alcohol",
    "obesity",
]

NAIVE_TARGET = (
    "smoking == 1 AND age >= 60"
)


def infection_frequency_per_person(runs):
    """
    Raw fraction of epidemic runs in which each person
    becomes infected.
    """

    infected = np.stack(
        [
            run.infection_time >= 0
            for run in runs
        ]
    )

    return infected.mean(axis=0)


def mean_offspring_per_person(
    graph,
    runs,
    susceptibility,
    edge_multipliers,
):
    """
    Raw mean number of secondary infections caused by each
    person, conditional on being included as an infected
    Model B observation.

    No opportunity normalisation is applied.
    """

    obs = model_b_observations(
        graph,
        runs,
        susceptibility=susceptibility,
        edge_multipliers=edge_multipliers,
    )

    offspring_sum = np.bincount(
        obs.node_index,
        weights=obs.offspring,
        minlength=N_NODES,
    )

    infected_count = np.bincount(
        obs.node_index,
        minlength=N_NODES,
    )

    mean_offspring = np.divide(
    offspring_sum,
    infected_count,
    out=np.full(
        N_NODES,
        np.nan,
        dtype=np.float64,
    ),
    where=infected_count > 0,
)

    return mean_offspring, infected_count

In [54]:
adversarial_naive_people = (
    adversarial_naive_people[
        adversarial_naive_people[
            "mean_offspring"
        ].notna()
    ]
    .copy()
)

In [55]:
# ============================================================
# Naive susceptibility baseline:
# DEAD-END stress test
#
# Truth:
# susceptibility RR = 1.50
#
# Model A:
# correctly recovered the target at rank 1.
#
# Naive baseline:
# uses only infection frequency.
# ============================================================

dead_end_infection_frequency = (
    infection_frequency_per_person(
        dead_end_runs_filtered
    )
)

dead_end_naive_people = (
    people.copy()
)

dead_end_naive_people[
    "infection_frequency"
] = dead_end_infection_frequency


dead_end_naive_results = (
    search_subgroups(
        dead_end_naive_people,
        target_column=(
            "infection_frequency"
        ),
        categorical_columns=(
            NAIVE_CATEGORICAL_COLUMNS
        ),
        age_column="age",
        age_thresholds=range(
            20,
            81,
            5,
        ),
        min_size=200,
        max_depth=2,
        direction="high",
    )
)


dead_end_naive_target = (
    dead_end_naive_results[
        dead_end_naive_results[
            "description"
        ]
        == NAIVE_TARGET
    ]
)


print(
    "DEAD-END NAIVE INFECTION-FREQUENCY BASELINE"
)

print(
    "Exact planted target rank:",
    dead_end_naive_target.index[0] + 1,
)

print(
    "Target subgroup mean infection frequency:",
    dead_end_naive_target.iloc[0][
        "mean_residual"
    ],
)

print(
    "Complement mean infection frequency:",
    dead_end_naive_target.iloc[0][
        "rest_mean_residual"
    ],
)

print(
    "Naive quality:",
    dead_end_naive_target.iloc[0][
        "quality"
    ],
)

print(
    "\nModel A target rank:",
    dead_end_a_target.index[0] + 1,
)

print(
    "Model A estimated susceptibility RR:",
    dead_end_a_target.iloc[0][
        "rate_ratio"
    ],
)

display(
    dead_end_naive_results.head(10)
)

DEAD-END NAIVE INFECTION-FREQUENCY BASELINE
Exact planted target rank: 340
Target subgroup mean infection frequency: 0.3211886304909561
Complement mean infection frequency: 0.4625104649847268
Naive quality: -0.04815322472102877

Model A target rank: 1
Model A estimated susceptibility RR: 1.5122663812450512


,depth,description,size,support,mean_residual,rest_mean_residual,difference,quality
0,1,age < 70,9363,0.9363,0.462228,0.209089,0.253138,0.244943
1,1,age < 65,8723,0.8723,0.478030,0.228011,0.250020,0.233511
2,1,age < 60,8040,0.8040,0.484546,0.288408,0.196138,0.175869
3,1,age < 55,7358,0.7358,0.495826,0.307623,0.188203,0.161439
4,1,age < 50,6714,0.6714,0.507021,0.321634,0.185387,0.151904
5,2,obesity == 0 AND age < 65,8102,0.8102,0.477883,0.310443,0.167441,0.150715
6,2,obesity == 0 AND age < 60,7464,0.7464,0.484310,0.333651,0.150659,0.130161
7,2,obesity == 0 AND age < 55,6827,0.6827,0.495850,0.339067,0.156783,0.129543
8,2,obesity == 0 AND age < 50,6238,0.6238,0.506811,0.345439,0.161373,0.127454
9,1,age < 45,6047,0.6047,0.509644,0.348902,0.160742,0.124997


In [56]:
# ============================================================
# Naive infectiousness baseline:
# ADVERSARIAL SPREADER stress test
#
# Truth:
# susceptibility RR = 0.60
# infectiousness RR = 1.75
#
# Compare raw offspring with opportunity-adjusted Model B.
# ============================================================

(
    adversarial_mean_offspring,
    adversarial_infected_counts,
) = mean_offspring_per_person(
    graph,
    adversarial_runs_filtered,
    susceptibility=(
        adversarial_params.susceptibility
    ),
    edge_multipliers=edge_multipliers,
)


print(
    "People never appearing as naturally infected "
    "Model B observations:",
    int(
        (
            adversarial_infected_counts == 0
        ).sum()
    ),
)


adversarial_naive_people = (
    people.copy()
)

adversarial_naive_people[
    "mean_offspring"
] = adversarial_mean_offspring


adversarial_naive_offspring_results = (
    search_subgroups(
        adversarial_naive_people,
        target_column="mean_offspring",
        categorical_columns=(
            NAIVE_CATEGORICAL_COLUMNS
        ),
        age_column="age",
        age_thresholds=range(
            20,
            81,
            5,
        ),
        min_size=200,
        max_depth=2,
        direction="high",
    )
)


adversarial_naive_offspring_target = (
    adversarial_naive_offspring_results[
        adversarial_naive_offspring_results[
            "description"
        ]
        == NAIVE_TARGET
    ]
)


print(
    "\nADVERSARIAL NAIVE OFFSPRING BASELINE"
)

print(
    "Exact planted target rank:",
    adversarial_naive_offspring_target.index[0]
    + 1,
)

print(
    "Target mean offspring:",
    adversarial_naive_offspring_target.iloc[0][
        "mean_residual"
    ],
)

print(
    "Complement mean offspring:",
    adversarial_naive_offspring_target.iloc[0][
        "rest_mean_residual"
    ],
)

print(
    "Naive quality:",
    adversarial_naive_offspring_target.iloc[0][
        "quality"
    ],
)

print(
    "\nModel B target rank:",
    adversarial_b_target.index[0] + 1,
)

print(
    "Model B estimated infectiousness RR:",
    adversarial_b_target.iloc[0][
        "rate_ratio"
    ],
)

display(
    adversarial_naive_offspring_results.head(10)
)

People never appearing as naturally infected Model B observations: 1

ADVERSARIAL NAIVE OFFSPRING BASELINE
Exact planted target rank: 262
Target mean offspring: 0.8490259553014111
Complement mean offspring: 0.9116792465527513
Naive quality: -0.021349205401703403

Model B target rank: 1
Model B estimated infectiousness RR: 1.7594388604487645


,depth,description,size,support,mean_residual,rest_mean_residual,difference,quality
0,1,age < 70,9363,0.936394,0.929931,0.528616,0.401315,0.388342
1,1,age < 65,8723,0.872387,0.953330,0.569938,0.383392,0.358095
2,2,obesity == 0 AND age < 65,8102,0.810281,0.953360,0.695316,0.258045,0.232281
3,1,age < 60,8040,0.804080,0.951804,0.709872,0.241932,0.216941
4,1,age < 55,7358,0.735874,0.969390,0.723350,0.246040,0.211060
5,1,age < 50,6714,0.671467,0.986407,0.736805,0.249603,0.204532
6,2,obesity == 0 AND age < 70,8699,0.869987,0.929647,0.735496,0.194151,0.181091
7,2,obesity == 0 AND age < 50,6238,0.623862,0.986052,0.768983,0.217069,0.171452
8,2,obesity == 0 AND age < 55,6827,0.682768,0.969048,0.765274,0.203774,0.168378
9,1,age < 45,6047,0.604760,0.987750,0.776876,0.210874,0.163989


In [57]:
# ============================================================
# Naive vs contact-adjusted comparison
# ============================================================

naive_baseline_summary = pd.DataFrame(
    [
        {
            "scenario":
                "Dead-end",

            "mechanism":
                "Susceptibility",

            "true_effect":
                DEAD_END_SUSCEPTIBILITY,

            "naive_target":
                "Infection frequency",

            "naive_target_rank":
                int(
                    dead_end_naive_target.index[0]
                    + 1
                ),

            "adjusted_model":
                "Model A",

            "adjusted_target_rank":
                int(
                    dead_end_a_target.index[0]
                    + 1
                ),

            "adjusted_estimate":
                float(
                    dead_end_a_target.iloc[0][
                        "rate_ratio"
                    ]
                ),
        },

        {
            "scenario":
                "Adversarial spreader",

            "mechanism":
                "Infectiousness",

            "true_effect":
                ADVERSARIAL_INFECTIOUSNESS,

            "naive_target":
                "Mean offspring",

            "naive_target_rank":
                int(
                    adversarial_naive_offspring_target
                    .index[0]
                    + 1
                ),

            "adjusted_model":
                "Model B",

            "adjusted_target_rank":
                int(
                    adversarial_b_target.index[0]
                    + 1
                ),

            "adjusted_estimate":
                float(
                    adversarial_b_target.iloc[0][
                        "rate_ratio"
                    ]
                ),
        },
    ]
)

display(
    naive_baseline_summary
)

,scenario,mechanism,true_effect,naive_target,naive_target_rank,adjusted_model,adjusted_target_rank,adjusted_estimate
0,Dead-end,Susceptibility,1.50,Infection frequency,340,Model A,1,1.512266
1,Adversarial spreader,Infectiousness,1.75,Mean offspring,262,Model B,1,1.759439
